## Setup

In [1]:
import json
import difflib
import re
from collections import Counter, defaultdict

import pandas as pd
import spacy
from spacy.language import Language

# Eigenes NER-Modell (nach dem Training)
nlp_ner = spacy.load("output/model-best")

pd.set_option("display.max_colwidth", 90)

# Bestes verfügbares deutsches Modell laden (lg > md > sm)
MODELL_KANDIDATEN = ["de_core_news_lg", "de_core_news_md", "de_core_news_sm"]
nlp = None
for _name in MODELL_KANDIDATEN:
    try:
        nlp = spacy.load(_name)
        print(f"spaCy-Modell geladen: {_name}")
        break
    except OSError:
        continue
if nlp is None:
    raise OSError(
        "Kein deutsches spaCy-Modell gefunden. Installation z. B. mit:\n"
        "    python -m spacy download de_core_news_lg"
    )

from pathlib import Path


spaCy-Modell geladen: de_core_news_lg


### Satzgrenzen an Anführungszeichen korrigieren

In [ ]:
ANFUEHRUNGSZEICHEN = {'"', "„", "“", "”", "»", "«", "›", "‹", "‚", "‘", "’"}
SATZSCHLUSSZEICHEN = {".", "!", "?", "…"}


def quote_grenzen(doc):
    for i in range(1, len(doc)):
        tok = doc[i]
        if tok.text not in ANFUEHRUNGSZEICHEN:
            continue
        haengt_links = doc[i - 1].whitespace_ == ""
        haengt_rechts = tok.whitespace_ == "" and i + 1 < len(doc)
        if haengt_links:
            tok.is_sent_start = False
        elif haengt_rechts and doc[i - 1].text in SATZSCHLUSSZEICHEN:
            tok.is_sent_start = True
            doc[i + 1].is_sent_start = False
    return doc


if not Language.has_factory("quote_grenzen"):
    Language.component("quote_grenzen", func=quote_grenzen)
if "parser" in getattr(nlp, "pipe_names", []) and "quote_grenzen" not in nlp.pipe_names:
    nlp.add_pipe("quote_grenzen", before="parser")
print("Pipeline:", getattr(nlp, "pipe_names", "(mock)"))

Pipeline: ['tok2vec', 'tagger', 'morphologizer', 'quote_grenzen', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']


## Schnittstelle zur Coreference Resolution

In [ ]:
def resolve_coreferences(text):
    raise NotImplementedError(
        "Modul nicht geladen -- zuerst die Import-Zelle in Schritt 2 ausfuehren.")


def resolve_coreferences_protokolliert(text):
    raise NotImplementedError(
        "Modul nicht geladen -- zuerst die Import-Zelle in Schritt 2 ausfuehren.")

## Schritt 1 — Text einlesen, normalisieren & Figuren per NER identifizieren

In [ ]:
def normalisiere_text(text):
    text = re.sub(r"(\w)-\n([a-zäöüß])", r"\1\2", text)
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


### Schritt 1a: NER-Figuren mit Satzkontext

In [ ]:
#===========================================================================
# Schritt 1: NER-Figuren MIT Satzkontext
#===========================================================================

def _token_merkmale(token):
    return {
        "text": token.text,
        "lemma": token.lemma_,
        "pos": token.pos_,
        "tag": token.tag_,
        "dep": token.dep_,
        "kasus": list(token.morph.get("Case")),
        "numerus": list(token.morph.get("Number")),
    }


def _span_merkmale(rohform, span):
    if span is None or len(span) == 0:
        #Fallback: kein Span im Haupt-Doc gefunden (Tokenisierungsdifferenz)
        return {
            "rohform": rohform,
            "lemma": lemmatisiere_name(rohform),
            "kopf_lemma": lemmatisiere_name(rohform).split()[-1],
            "kopf_pos": None,
            "numerus": None,
            "kasus": None,
            "tokens": [],
            "im_kontext": False,
        }

    kopf = span.root
    tokens = [_token_merkmale(t) for t in span]
    numerus = kopf.morph.get("Number")
    kasus = kopf.morph.get("Case")
    return {
        "rohform": rohform,
        #Lemma der gesamten Rohform, Token fuer Token aus dem Satzkontext
        "lemma": " ".join(t.lemma_ for t in span),
        "kopf_lemma": kopf.lemma_,
        "kopf_pos": kopf.pos_,
        "numerus": numerus[0] if numerus else None,
        "kasus": kasus[0] if kasus else None,
        "tokens": tokens,
        "im_kontext": True,
    }


def extrahiere_figuren_ner_detail(doc):
    doc_ner = nlp_ner(doc.text)
    zaehler = Counter()
    detail = []

    for ent in doc_ner.ents:
        if ent.label_ != "FIGUR":
            continue
        rohform = ent.text.strip()
        if not rohform:
            continue
        zaehler[rohform] += 1
        span = doc.char_span(ent.start_char, ent.end_char,
                             alignment_mode="expand")
        merkmale = _span_merkmale(rohform, span)
        merkmale["start_char"] = ent.start_char
        detail.append(merkmale)

    return zaehler, detail


def extrahiere_figuren_ner(satz):
    doc_ner = nlp_ner(satz)
    zaehler = Counter()
    for ent in doc_ner.ents:
        if ent.label_ == "FIGUR":
            zaehler[ent.text.strip()] += 1
    return zaehler


def lemmatisiere_name(name):
    return " ".join(t.lemma_ for t in nlp(name))

### Schritt 1b: Plausibilitätsschranke gegen NER-Fehlerkennungen


In [ ]:
MIN_EVIDENZ_FIGURENNOMEN = 4

#Muster für Fussnotenmarker aus Gutenberg-Rohtexten ("Verbeugungen[3")
FUSSNOTE_MUSTER = re.compile(r"\[\d+\]?$|\[\d+")

#Markup- und Satzzeichenreste am Wortrand ("~Daumesdick~.", "Finnin.+", "Yoro-Zeit.«")
#unsaubere Figurennamen aufgrund von Textformnatierungen
RANDZEICHEN_MUSTER = re.compile(r"^[^\wÄÖÜäöüß]+|[^\wÄÖÜäöüß]+$")


def bereinige_rohform(rohform):
    bereinigt = FUSSNOTE_MUSTER.sub("", rohform).strip()
    grund = "fussnote_entfernt" if bereinigt != rohform else None

    ohne_rand = RANDZEICHEN_MUSTER.sub("", bereinigt).strip()
    if ohne_rand != bereinigt:
        bereinigt = ohne_rand
        grund = "randzeichen_entfernt" if grund is None else "fussnote_und_randzeichen"

    if not bereinigt:
        return None, "nur_markup"
    return bereinigt, grund or "unveraendert"


def _hat_genitivsignal(tok):
    text = tok.get("text", "")
    if len(text) > 2 and text[:1].isupper() and (text.endswith("s") or text.endswith("'")):
        return True
    if tok.get("pos") == "PROPN" and "Gen" in (tok.get("kasus") or []):
        return True
    return False


def baue_ner_evidenz(ergebnisse):
    evidenz = Counter()
    for res in ergebnisse:
        for rohform, anzahl in res.get("figuren_ner", {}).items():
            evidenz[rohform.strip().lower()] += anzahl
    return evidenz


def pruefe_mehrwortname(merkmale, evidenz, min_evidenz=None):
    min_evidenz = MIN_EVIDENZ_FIGURENNOMEN if min_evidenz is None else min_evidenz
    tokens = merkmale.get("tokens") or []
    rohform = merkmale["rohform"]

    if len(tokens) < 2:
        return "behalten", None, "einwort"

    def ev(text):
        return evidenz.get(text.strip().lower(), 0)

    #--- Regel 1: Genitivschranke --------------------------------
    if not any(_hat_genitivsignal(t) for t in tokens[:-1]):
        #--- Regel 2: Praefix-Kuerzung -------------------------------------
        erstes = tokens[0]["text"]
        rest = " ".join(t["text"] for t in tokens[1:])
        if ev(erstes) == 0 and ev(rest) > 0 and ev(rest) > ev(rohform):
            return ("kuerzen", rest,
                    f"praefix_ohne_evidenz ({erstes}=0, {rest}={ev(rest)} "
                    f"gegen ganze Form {ev(rohform)})")
        return "behalten", None, "kein_genitivmuster"

    kopf = tokens[-1]
    kopf_pos = kopf["pos"]
    if kopf_pos == "PROPN":
        return "behalten", None, "kopf_ist_eigenname"

    kopf_evidenz = max(ev(kopf["text"]), ev(kopf["lemma"]))
    if kopf_evidenz >= min_evidenz:
        return "behalten", None, f"kopf_als_figur_belegt ({kopf_evidenz}x)"
    return "verwerfen", None, f"genitiv_sachnomen (Kopf {kopf_evidenz}x belegt)"


print(f"Plausibilitätsschranke: evidenzbasiert, Schwelle "
      f"MIN_EVIDENZ_FIGURENNOMEN = {MIN_EVIDENZ_FIGURENNOMEN} "
      f"(keine Wortlisten)")

Plausibilitätsschranke: evidenzbasiert, Schwelle MIN_EVIDENZ_FIGURENNOMEN = 4 (keine Wortlisten)


### Schritt 1c: Figurenindex mit getrenntem Schlüssel und Anzeigenamen

In [ ]:
def _normform(s):
    return s.strip().lower()


def baue_figurenindex(figuren_detail, aliase=None, bereinigen=True):
    aliase_norm = {_normform(k): v for k, v in (aliase or {}).items()}

    schluessel_info = {}
    bereinigt_log = []

    for merkmale in figuren_detail:
        rohform = merkmale["rohform"]

        #--- Kontextfreie Bereinigung (Fussnotenmarker) --------------------
        if bereinigen:
            neu, grund = bereinige_rohform(rohform)
            if neu is None:
                bereinigt_log.append({"rohform": rohform, "grund": grund,
                                      "ziel": None})
                continue
            if neu != rohform:
                #Das Lemma MUSS mitbereinigt werden, denn aus ihm entsteht der
                #Schluessel. Ohne diesen Schritt trug "Nishikaze[2" seinen
                #Fussnotenmarker in den Schluessel weiter, obwohl die Rohform
                #bereits korrigiert war.
                lemma_neu, _ = bereinige_rohform(merkmale.get("lemma") or neu)
                bereinigt_log.append({"rohform": rohform, "grund": grund,
                                      "ziel": neu})
                merkmale = dict(merkmale, rohform=neu, bereinigt_von=rohform,
                                lemma=lemma_neu or neu)
                rohform = neu

        #--- Schluessel bestimmen ------------------------------------------
        alias_ziel = aliase_norm.get(_normform(rohform))
        if alias_ziel:
            schluessel = _normform(alias_ziel)
        else:
            #Lemma aus dem Satzkontext; Fallback auf die Rohform
            schluessel = _normform(merkmale.get("lemma") or rohform)
        if not schluessel:
            schluessel = _normform(rohform)

        eintrag = schluessel_info.setdefault(schluessel, {
            "formen": Counter(),
            "anzahl": 0,
            "merkmale": [],
            "numerus": Counter(),
            "alias_ziel": None,
        })
        eintrag["formen"][rohform] += 1
        eintrag["anzahl"] += 1
        eintrag["merkmale"].append(merkmale)
        if merkmale.get("numerus"):
            eintrag["numerus"][merkmale["numerus"]] += 1
        if alias_ziel:
            eintrag["alias_ziel"] = alias_ziel

    #--- Nachschlage-Index aufbauen ----------------------------------------
    #Indiziert werden ALLE Rohformen UND ihre Lemmata, damit ist_figur die
    #Oberflaechenform im Text spaeter wiederfindet.
    index = {}
    for schluessel, eintrag in schluessel_info.items():
        index.setdefault(schluessel, schluessel)
        for rohform in eintrag["formen"]:
            index.setdefault(_normform(rohform), schluessel)
        for merkmale in eintrag["merkmale"]:
            if merkmale.get("lemma"):
                index.setdefault(_normform(merkmale["lemma"]), schluessel)
            for tok in merkmale.get("tokens", []):
                #Teile mehrteiliger Namen: "Koenig Drosselbart" -> "drosselbart"
                if len(tok["text"]) > 3 and tok["pos"] in {"NOUN", "PROPN"}:
                    index.setdefault(_normform(tok["text"]), schluessel)
                    index.setdefault(_normform(tok["lemma"]), schluessel)
        for teil in schluessel.split():
            if len(teil) > 3:
                index.setdefault(teil, schluessel)

    return index, schluessel_info, bereinigt_log


#---------------------------------------------------------------------------
# Anzeigename: aus allen beobachteten Oberflaechenformen eines Schluessels
#---------------------------------------------------------------------------
def waehle_anzeigenamen(schluessel, formen, merkmale_liste=None):
    """
    Waehlt den lesbaren Namen zu einem Schluessel nach fester Rangfolge:
      1. Form, die im Text als Nominativ Singular belegt ist
      2. Form, deren Kleinschreibung dem Schluessel (= Lemma) entspricht
      3. haeufigste Oberflaechenform
    Bei Gleichstand entscheiden Haeufigkeit und Alphabetik, damit zwei
    Laeufe denselben Namen ergeben.

    Diese Funktion wird NACH dem Batch ueber alle Maerchen hinweg aufgerufen.
    Genau darin liegt die Reparatur von Befund 1: der Name haengt nicht mehr
    davon ab, welcher Kasus in einem einzelnen Maerchen dominiert.
    """
    if not formen:
        return schluessel

    nominativ_singular = {
        m["rohform"] for m in (merkmale_liste or [])
        if m.get("kasus") == "Nom" and m.get("numerus") == "Sing"
    }

    def rang(form):
        return (
            0 if form in nominativ_singular else 1,   #1. Nominativ Singular
            0 if _normform(form) == schluessel else 1,  #2. Lemmaform
            -formen[form],                            #3. Haeufigkeit
            form,                                     #4. Alphabetik
        )

    return min(formen, key=rang)

### Alias-Tabelle

In [ ]:
ALIASE = {
    #--- Form eines Eintrags (bewusst deaktiviert) --------------------------
}
print(f"{len(ALIASE)} Alias-Einträge geladen"
      + (": " + ", ".join(f"{k} -> {v}" for k, v in ALIASE.items())
         if ALIASE else " — Tabelle leer, Mechanismus aktiv (V5.12)"))


0 Alias-Einträge geladen — Tabelle leer, Mechanismus aktiv (V5.12)


## Schritt 2 — Coreference Resolution & Instanz-Disambiguierung laden

In [ ]:
from pipeline.coref_sliding_window import (
    resolve_coreferences,
    resolve_coreferences_protokolliert,
)


from pipeline import instanz_disambiguierung as inst


/home/timc/Workspace/fairytale-personality-profiling/venv_cuda_test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Geräte-Check] Parameter auf CPU: 0, Buffer auf CPU: 0
[Geräte-Check] Evaluationsmodus: True, aktive Dropout-Module: 0


## Schritt 3 — Original-Text in Sätze aufteilen (Satz-IDs vergeben)

In [ ]:
def repariere_satzspannen(spans): # Sicherheitsnetz nach dem Parsen
    spans = list(spans)
    repariert = []
    i = 0
    while i < len(spans):
        span = spans[i]
        doc = span.doc
        # 1) fuehrende schliessende Anfuehrungszeichen nach links verschieben
        while (len(span) > 0 and span[0].text in ANFUEHRUNGSZEICHEN
               and span.start > 0 and doc[span.start - 1].whitespace_ == ""
               and repariert):
            repariert[-1] = doc[repariert[-1].start: span.start + 1]
            span = doc[span.start + 1: span.end]
        if len(span) == 0:
            i += 1
            continue
        # 2) Saetze ohne alphabetisches/numerisches Material verschmelzen
        if all(not t.is_alpha and not t.like_num for t in span):
            if span[-1].whitespace_ == "" and i + 1 < len(spans):
                spans[i + 1] = doc[span.start: spans[i + 1].end]
            elif repariert:
                repariert[-1] = doc[repariert[-1].start: span.end]
            else:
                repariert.append(span)
            i += 1
            continue
        repariert.append(span)
        i += 1
    return repariert


## Schritt 4 — Satzmapping: Original ↔ Coref

In [ ]:
#Paare unterhalb dieser Tokenaehnlichkeit gelten als Fehlpaarung, Wert wurde gemessen
MIN_AEHNLICHKEIT_MAPPING = 0.35


def tokenisiere_mit_satz(saetze):
    """Zerlegt Saetze in Tokens und merkt sich je Token den Satzindex."""
    tokens, satz_von_token = [], []
    for i, satz in enumerate(saetze):
        for t in re.findall(r"\w+|[^\w\s]", satz, flags=re.UNICODE):
            tokens.append(t)
            satz_von_token.append(i)
    return tokens, satz_von_token


def tokenaehnlichkeit(a, b):
    """Anteil uebereinstimmender Tokens zweier Saetze."""
    ta = re.findall(r"\w+", (a or "").lower(), flags=re.UNICODE)
    tb = re.findall(r"\w+", (b or "").lower(), flags=re.UNICODE)
    if not ta and not tb:
        return 1.0
    if not ta or not tb:
        return 0.0
    return difflib.SequenceMatcher(None, ta, tb, autojunk=False).ratio()


def baue_satzmapping(saetze_original, saetze_coref,
                     min_aehnlichkeit=MIN_AEHNLICHKEIT_MAPPING):
    """
    Ordnet Original- und Coref-Saetze ueber die gemeinsame Tokenfolge zu.

    Rueckgabe: Liste von Eintraegen mit
      id            Satznummer im Original
      original      Satztext des Originals
      coref         zugeordneter Coref-Satz, None bei Luecke oder Verwurf
      coref_id      Satznummer im Coref-Text (auch bei Verwurf erhalten)
      aehnlichkeit  Tokenaehnlichkeit des Paares
      deckung       Anteil der Satztokens, die im Coref-Text wiedergefunden wurden
      verworfen     True, wenn die Aehnlichkeit unter der Schwelle liegt
    """
    tok_o, satz_o = tokenisiere_mit_satz(saetze_original)
    tok_c, satz_c = tokenisiere_mit_satz(saetze_coref)

    #uebereinstimmende Bloecke sind monoton geordnet -- daraus entsteht eine
    #Abbildung der Tokenpositionen, unabhaengig von den Satzgrenzen
    treffer = {}
    for i, j, n in difflib.SequenceMatcher(None, tok_o, tok_c).get_matching_blocks():
        for k in range(n):
            treffer[i + k] = j + k

    #je Originalsatz auszaehlen, in welchem Coref-Satz seine Tokens liegen
    stimmen = [dict() for _ in saetze_original]
    for pos_o, pos_c in treffer.items():
        zaehler = stimmen[satz_o[pos_o]]
        ziel = satz_c[pos_c]
        zaehler[ziel] = zaehler.get(ziel, 0) + 1

    tokens_je_satz = Counter(satz_o)
    mapping = []
    for i, satz in enumerate(saetze_original):
        zaehler = stimmen[i]
        if not zaehler:
            #kein einziges Token wiedergefunden -- echte Luecke
            mapping.append({"id": i, "original": satz, "coref": None,
                            "coref_id": None, "aehnlichkeit": None,
                            "deckung": 0.0, "verworfen": False})
            continue

        #Mehrheit entscheidet; bei Gleichstand der naeherliegende Satz
        ziel = max(zaehler, key=lambda k: (zaehler[k], -abs(k - i)))
        deckung = zaehler[ziel] / max(1, tokens_je_satz[i])
        wert = tokenaehnlichkeit(satz, saetze_coref[ziel])
        zu_schwach = wert < min_aehnlichkeit
        mapping.append({
            "id": i,
            "original": satz,
            "coref": None if zu_schwach else saetze_coref[ziel],
            "coref_id": ziel,
            "aehnlichkeit": round(wert, 4),
            "deckung": round(deckung, 4),
            "verworfen": zu_schwach,
        })
    return mapping


## Schritt 5 — POS-Tagging (Original)

## Schritt 6 — POS-Filterung: nur Sätze mit Adjektiven


In [ ]:
ADJ_TAGS        = {"ADJA", "ADJD"}                   # STTS: attributiv / prädikativ-adverbial


def ist_adjektiv(token):
    """Erkennt Adjektive ueber POS oder STTS-Tag (ADJD wird gelegentlich als ADV getaggt)."""
    return token.pos_ == "ADJ" or token.tag_ in ADJ_TAGS


## Schritt 7 — Dependency Parsing: welche Adjektive gehören zu welchen Figuren


In [ ]:
ATTRIBUTIV_DEPS = {"nk", "amod"}                     # "die kalte Stiefmutter"
SUBJEKT_DEPS    = {"sb", "sbp", "nsubj", "nsubj:pass"}
OBJEKT_DEPS     = {"oa", "obj"}                      # "machte sie traurig"
KONJUNKT_DEPS   = {"cj", "cd", "conj", "cc"}         # "kalt UND herzlos"
RELATIVPRON_TAGS = {"PRELS", "PRELAT"}               # STTS: Relativpronomen
RELATIVSATZ_DEPS = {"rc", "relcl", "acl:relcl"}      # TIGER | UD


def _konjunktions_basis(token, max_schritte=19):
    """Klettert von einem Konjunkt ('herzlos') zum ersten Konjunkt ('kalt') hoch."""
    t = token
    schritte = 0
    while t.dep_ in KONJUNKT_DEPS and t.head != t and schritte < max_schritte:
        t = t.head
        schritte += 1
    return t


def _verb_mit_subjekt(verb, max_schritte=19):
    """
    Sucht das Subjekt eines Verbs
    Rückgabe: (Verb mit Subjekt, Subjekt | None)
    """
    t = verb
    schritte = 0
    while schritte < max_schritte:
        subjekt = next((k for k in t.children if k.dep_ in SUBJEKT_DEPS), None)
        if subjekt is not None:
            return t, subjekt
        if t.dep_ in KONJUNKT_DEPS and t.head != t:
            t = t.head
            schritte += 1
            continue
        return t, None
    return t, None


def _aufloese_relativpronomen(pron, max_schritte=19):
    """
    'Töchter, DIE schön waren' -> Antezedens 'Töchter'.
    """
    if pron.tag_ not in RELATIVPRON_TAGS:
        return pron
    t = pron.head
    schritte = 0
    while t.dep_ not in RELATIVSATZ_DEPS and t.head != t and schritte < max_schritte:
        t = t.head
        schritte += 1
    if t.dep_ in RELATIVSATZ_DEPS and t.head.pos_ in {"NOUN", "PROPN"}:
        return t.head
    return pron


#Kopulaverben: nur hier prädiziert das Adjektiv über das Subjekt
KOPULA_LEMMATA = {"sein", "werden", "bleiben", "scheinen", "wirken",
                  "aussehen", "heißen", "dünken"}

#Geschlossene Adverbklassen ohne Figureneigenschaft. Die Mengen SPERREN
#NICHTS. Sie ordnen den Beleg lediglich einer Klasse zu. Gebildet wurden sie
#aus grammatischen Klassen und ausdrüüklich NICHT aus den Fehlern der
#Stichprobe, weil eine dort abgelesene Liste ihre eigene Wirkung bestätigen
#würde. Adverbien der Art und Weise (rasch, leise, gehässig, mühsam)
#fehlen bewusst, da sie laut Auszeichnung eine Figur charakterisieren.
ADVERB_ZEIT = {
    "heute", "gestern", "morgen", "morgens", "abends", "nachts", "mittags",
    "täglich", "tagtäglich", "wöchentlich", "monatlich", "jährlich",
    "früh", "früher", "spät", "später", "damals", "jetzt", "nun", "bald",
    "sofort", "sogleich", "alsbald", "endlich", "zuletzt", "zuerst", "wieder",
    "immer", "oft", "selten", "nie", "niemals", "stets", "fortwährend",
    "lange", "längst", "vorher", "nachher", "seither", "künftig",
}
ADVERB_GRAD = {
    "sehr", "ganz", "gänzlich", "völlig", "vollständig", "ziemlich", "äußerst",
    "außerordentlich", "besonders", "überaus", "weit", "viel", "mehr", "weniger",
    "allzu", "recht", "fast", "beinahe", "kaum", "genug", "ebenso", "höchst",
    "gar", "garnicht", "überhaupt", "sogar", "bloß", "nur",
}
ADVERB_RICHTUNG = {
    "nah", "näher", "fern", "weiter", "tief", "tiefer", "hoch", "höher",
    "umher", "herum", "fort", "weg", "hin", "her", "oben", "unten", "vorn",
    "hinten", "hinaus", "herein", "empor", "hinab", "voran", "zurück",
}
ADVERB_SATZ = {
    "zufällig", "tatsächlich", "wirklich", "wahrhaftig", "freilich", "natürlich",
    "vielleicht", "gewiss", "sicherlich", "wahrscheinlich", "offenbar",
    "angeblich", "leider", "glücklicherweise", "kurz", "nämlich", "eigentlich",
}
ADVERB_OHNE_FIGURENBEZUG = (ADVERB_ZEIT | ADVERB_GRAD
                            | ADVERB_RICHTUNG | ADVERB_SATZ)

SPERRE_ADVERBIAL = True
GESPERRTE_KLASSEN = {"adverbial_ohne_figurenbezug"} if SPERRE_ADVERBIAL else set()
GRUND_ADVERBIAL = "adverbial_ohne_figurenbezug"

BEZUG_KLASSE_SICHER = {
    #0 Bezugsfehler bei 285 geprüften Belegen (obere Schranke 1,3 Prozent)
    "attributiv": True,
    #echte Praedikation über Kopula oder Dependenz 'pd'
    "praedikativ": True,
    #Objekt als Ziel, nur bei 'machen' belastbar
    "objektspraedikativ": True,
    #2 Fehler bei 7 Belegen; Basis zu klein für eine Aussage
    "relativsatz": False,
    #adverbial am Vollverb, Art und Weise; getrennt nicht nachgemessen
    "artangabe": False,
    #Zeit-, Grad-, Richtungs- und Satzadverbien: tragen keine Figureneigenschaft
    "adverbial_ohne_figurenbezug": False,
}


def _leere_bewertung():
    return {"bezug_regel": None, "bezug_klasse": None, "bezug_sicher": None,
            "bezug_nominalkopf": False,
            #wird gesetzt, wenn die Klasse gesperrt ist
            "bezug_gesperrt": False, "grund_regel": None}


def finde_bezugswort_bewertet(adj, max_schritte=10):
    """
    Wie 'finde_bezugswort', liefert zusätzlich die Einordnung des Belegs.
    Rueckgabe: (Token | None, Art der Konstruktion | None, Bewertung: dict)
    """
    bewertung = _leere_bewertung()
    basis = _konjunktions_basis(adj)

    bewertung["bezug_nominalkopf"] = bool(
        basis.head != basis
        and basis.head.pos_ in {"NOUN", "PROPN"}
        and basis.dep_ not in ATTRIBUTIV_DEPS
    )

    # 1) attributiv: Adjektiv haengt direkt am Nomen
    if basis.dep_ in ATTRIBUTIV_DEPS and basis.head.pos_ in {"NOUN", "PROPN", "PRON"}:
        bewertung.update(bezug_regel="attributiv", bezug_klasse="attributiv",
                         bezug_sicher=BEZUG_KLASSE_SICHER["attributiv"])
        return basis.head, "attributiv", bewertung

    # 2) praedikativ / adverbial: zum regierenden Verb hochklettern
    kopf = basis
    schritte = 0
    while kopf.pos_ not in {"VERB", "AUX"} and kopf.head != kopf and schritte < max_schritte:
        kopf = kopf.head
        schritte += 1

    if kopf.pos_ in {"VERB", "AUX"}:
        verb_zuerst = kopf
        kopf, subjekt = _verb_mit_subjekt(kopf)
        #Muss ich nochmal abchecken: Wurde über die cj/cd-Kette zu einem anderen Verb geklettert?
        ueber_koordination = kopf.i != verb_zuerst.i
        objekt = next((k for k in kopf.children if k.dep_ in OBJEKT_DEPS), None)
        #Bei 'machen' beschreibt das Adjektiv das Objekt: "machte SIE traurig"
        if kopf.lemma_ == "machen" and objekt is not None:
            bewertung.update(bezug_regel="objektspraedikativ_machen",
                             bezug_klasse="objektspraedikativ",
                             bezug_sicher=BEZUG_KLASSE_SICHER["objektspraedikativ"])
            return objekt, "objektspraedikativ", bewertung

        #Einordnung des Verbwegs
        if adj.lemma_.strip().lower() in ADVERB_OHNE_FIGURENBEZUG:
            klasse = "adverbial_ohne_figurenbezug"
        elif basis.dep_ == "pd" or kopf.lemma_.strip().lower() in KOPULA_LEMMATA:
            klasse = "praedikativ"
        else:
            klasse = "artangabe"

        #gesperrte Klassen werden markiert
        gesperrt = klasse in GESPERRTE_KLASSEN

        if subjekt is not None:
            antezedens = _aufloese_relativpronomen(subjekt)
            if antezedens is not subjekt:
                bewertung.update(bezug_regel="relativsatz",
                                 bezug_klasse="relativsatz",
                                 bezug_sicher=BEZUG_KLASSE_SICHER["relativsatz"],
                                 #ein Relativsatz ist nie adverbial gesperrt
                                 bezug_gesperrt=False, grund_regel=None)
                return antezedens, "relativsatz", bewertung
            bewertung.update(
                bezug_regel=("subjekt_verbkoordination" if ueber_koordination
                             else "subjekt_direkt"),
                bezug_klasse=klasse,
                bezug_sicher=BEZUG_KLASSE_SICHER[klasse],
                bezug_gesperrt=gesperrt,
                grund_regel=GRUND_ADVERBIAL if gesperrt else None,
            )
            return subjekt, "praedikativ/adverbial", bewertung
        if objekt is not None:
            bewertung.update(bezug_regel="objekt_ersatzweise",
                             bezug_klasse="objektspraedikativ",
                             bezug_sicher=BEZUG_KLASSE_SICHER["objektspraedikativ"],
                             #die ersatzweise Objektwahl fällt nicht unter die adverbiale Klasse und bleibt deshalb unberührt
                             bezug_gesperrt=False, grund_regel=None)
            return objekt, "objektspraedikativ", bewertung

    bewertung.update(bezug_regel="kein_treffer")
    return None, None, bewertung


def finde_bezugswort(adj, max_schritte=10):
    bezug, art, _ = finde_bezugswort_bewertet(adj, max_schritte)
    return bezug, art


def ist_figur(token, figurenindex):
    """Prüft, ob ein Token (über Lemma oder Textform) eine bekannte Figur ist."""
    if token is None:
        return None
    for kandidat in (token.lemma_, token.text):
        treffer = figurenindex.get(kandidat.strip().lower())
        if treffer:
            return treffer
    return None

In [ ]:
#Zuordnungszeile: Numerus und Kollektivmarkierung (additiv)

KOLLEKTIV_LEMMATA = {
    "beide", "alle", "viele", "mehrere", "einige", "saemtlich", "sämtlich",
    "jeder", "jede", "manche", "wenige", "zahlreich",
}


def bestimme_numerus(token):
    if token is None:
        return None, False

    werte = token.morph.get("Number")
    numerus = werte[0] if werte else None

    kollektiv = numerus == "Plur"
    if not kollektiv:
        for kind in token.children:
            if kind.pos_ == "NUM" or kind.lemma_.strip().lower() in KOLLEKTIV_LEMMATA:
                kollektiv = True
                break
    return numerus, kollektiv


def bestimme_figur(token, figurenindex):
    if token is None:
        return None
    for kandidat in (token.lemma_, token.text):
        treffer = figurenindex.get(kandidat.strip().lower())
        if treffer:
            return treffer
    return None


#Name bleibt als Alias erhalten, damit bestehender Code weiterläuft
ist_figur = bestimme_figur


def extrahiere_adjektiv_zuordnungen(span_oder_doc, satz_id, figurenindex,
                                    methode="original"):
    zeilen = []
    for tok in span_oder_doc:
        if not ist_adjektiv(tok):
            continue
        bezug, art, bewertung = finde_bezugswort_bewertet(tok)
        schluessel = bestimme_figur(bezug, figurenindex)
        numerus, kollektiv = bestimme_numerus(bezug)

        if bewertung.get("bezug_gesperrt"):
            schluessel = None
            status, grund = "offen", bewertung["grund_regel"]
        elif schluessel:
            status, grund = "zugeordnet", None
        elif bezug is None:
            status, grund = "offen", "kein_bezugswort"
        elif bezug.pos_ == "PRON":
            status, grund = "offen", "pronomen"
        else:
            status, grund = "offen", "bezugswort_keine_figur"

        zeilen.append({
            "satz_id": satz_id,
            "adjektiv": tok.text,
            "lemma": tok.lemma_,
            "konstruktion": art,
            "bezugswort": bezug.text if bezug is not None else None,
            "bezug_pos": bezug.pos_ if bezug is not None else None,
            "figur": schluessel,
            "methode": methode,
            "status": status,
            "grund_offen": grund,
            "figur_schluessel": schluessel,
            "numerus": numerus,
            "kollektiv": kollektiv,
            "bezug_regel": bewertung["bezug_regel"],
            "bezug_klasse": bewertung["bezug_klasse"],
            "bezug_sicher": bewertung["bezug_sicher"],
            "bezug_nominalkopf": bewertung["bezug_nominalkopf"],
            "bezug_gesperrt": bool(bewertung.get("bezug_gesperrt")),
            "bezug_token_i": (bezug.i - getattr(span_oder_doc, "start", 0)
                              if bezug is not None else None),
        })
    return zeilen


#Spaltenreihenfolge
SPALTEN_V4 = ["satz_id", "adjektiv", "lemma", "konstruktion", "bezugswort",
              "bezug_pos", "figur", "methode", "status", "grund_offen"]
SPALTEN = SPALTEN_V4 + ["figur_schluessel", "numerus", "kollektiv"]

SPALTEN = SPALTEN + ["coref_verknuepft", "coref_sicher", "coref_belegklasse",
                     "coref_cluster_anzahl"]

SPALTEN = SPALTEN + ["bezug_regel", "bezug_klasse", "bezug_sicher",
                     "bezug_nominalkopf"]
#Kennzeichnung der gesperrten Belege
SPALTEN = SPALTEN + ["bezug_gesperrt"]
#Token-Anker und Instanzfelder
SPALTEN = SPALTEN + ["bezug_token_i", "figur_rolle", "figur_instanz",
                     "figur_instanz_methode", "figur_instanz_evidenz",
                     "instanz_verdacht", "instanz_verdacht_zerfall",
                     "instanz_verdacht_grund"]


## Schritt 8 — Offene Fälle: Adjektive ohne zugeordnete Figur


## Schritt 9 — Pronomen über das Satzmapping auflösen & erneut prüfen


## Korpus laden — alle Märchen finden


In [ ]:
#Alle Märchen-Textdateien im Korpus (inkl. aller Unterordner) einsammeln
KORPUS_DIR = Path("Korpus")
DATEI_MUSTER = "*.txt"

if not KORPUS_DIR.exists():
    raise FileNotFoundError(
        f"Korpus-Ordner nicht gefunden: {KORPUS_DIR.resolve()}\n"
        "Bitte das Notebook aus dem Projekt-Stammverzeichnis starten "
        "(dort, wo der Ordner 'Korpus/' liegt)."
    )

#rglob durchsucht rekursiv ALLE Unterordner/Sammlungen
maerchen_dateien = []
for pfad in sorted(KORPUS_DIR.rglob(DATEI_MUSTER)):
    if not pfad.is_file():
        continue
    rel = pfad.relative_to(KORPUS_DIR)
    #Sammlung = erster Unterordner unter Korpus/ (z. B. "Grimm"); sonst "(Wurzel)"
    sammlung = rel.parts[0] if len(rel.parts) > 1 else "(Wurzel)"
    #Titel aus dem Dateinamen ableiten: "schneewittchen_text.txt" -> "schneewittchen"
    titel = re.sub(r"[_-]?text$", "", pfad.stem, flags=re.IGNORECASE) or pfad.stem
    maerchen_dateien.append({
        "quelle": pfad.as_posix(),
        "sammlung": sammlung,
        "titel": titel,
        "pfad": pfad,
    })

if not maerchen_dateien:
    raise FileNotFoundError(
        f"Keine {DATEI_MUSTER}-Dateien unter {KORPUS_DIR.resolve()} gefunden."
    )

anzahl_sammlungen = len({m["sammlung"] for m in maerchen_dateien})
print(f"{len(maerchen_dateien)} Märchen im Korpus gefunden "
      f"(über {anzahl_sammlungen} Sammlung(en)):\n")
for m in maerchen_dateien:
    print(f"  [{m['sammlung']}] {m['titel']}  ->  {m['quelle']}")

df_korpus = pd.DataFrame(
    [{k: v for k, v in m.items() if k != "pfad"} for m in maerchen_dateien]
)
df_korpus

80 Märchen im Korpus gefunden (über 4 Sammlung(en)):

  [Arabische Märchen] 1001 Nacht Epilog  ->  Korpus/Arabische Märchen/1001 Nacht Epilog.txt
  [Arabische Märchen] 1001 Nacht Prolog  ->  Korpus/Arabische Märchen/1001 Nacht Prolog.txt
  [Arabische Märchen] Geschichte des Prinzen Achmed und der Fee Pari Banu  ->  Korpus/Arabische Märchen/Geschichte des Prinzen Achmed und der Fee Pari Banu.txt
  [Arabische Märchen] Geschichte vom Fischer und dem Geiste  ->  Korpus/Arabische Märchen/Geschichte vom Fischer und dem Geiste.txt
  [Arabische Märchen] Geschichte vom Prinzen Chodadad  ->  Korpus/Arabische Märchen/Geschichte vom Prinzen Chodadad.txt
  [Arabische Märchen] Geschichte vom Zauberpferde  ->  Korpus/Arabische Märchen/Geschichte vom Zauberpferde.txt
  [Arabische Märchen] Geschichte vom versteinerten Prinzen  ->  Korpus/Arabische Märchen/Geschichte vom versteinerten Prinzen.txt
  [Arabische Märchen] Geschichte von Ali Baba und den vierzig Räubern  ->  Korpus/Arabische Märchen/Geschich

,quelle,sammlung,titel
0,Korpus/Arabische Märchen/1001 Nacht Epilog.txt,Arabische Märchen,1001 Nacht Epilog
1,Korpus/Arabische Märchen/1001 Nacht Prolog.txt,Arabische Märchen,1001 Nacht Prolog
2,Korpus/Arabische Märchen/Geschichte des Prinzen Achmed und der Fee Pari Banu.txt,Arabische Märchen,Geschichte des Prinzen Achmed und der Fee Pari Banu
3,Korpus/Arabische Märchen/Geschichte vom Fischer und dem Geiste.txt,Arabische Märchen,Geschichte vom Fischer und dem Geiste
4,Korpus/Arabische Märchen/Geschichte vom Prinzen Chodadad.txt,Arabische Märchen,Geschichte vom Prinzen Chodadad
...,...,...,...
75,Korpus/Japanische Märchen/Maorigashima.txt,Japanische Märchen,Maorigashima
76,Korpus/Japanische Märchen/Oschoo.txt,Japanische Märchen,Oschoo
77,Korpus/Japanische Märchen/Rai-taro.txt,Japanische Märchen,Rai-taro
78,Korpus/Japanische Märchen/Urashima Taro.txt,Japanische Märchen,Urashima Taro


## Kern: eine Pipeline pro Märchen

### Schritt 9b — Konfidenz der Coreference Resolution


In [ ]:
#Belegklassen des Coref-Moduls, geordnet von schwach nach stark
BELEGKLASSEN_RANG = {"langspanne": 0, "schwach": 1, "kopfposition": 2,
                     "eigenstaendig": 3}

#Endungen für den groben Stammvergleich, längste zuerst
KONFIDENZ_ENDUNGEN = ("ens", "ern", "en", "es", "ns", "er", "em", "e", "n", "s")


def konfidenz_stamm(wort):
    #Sehr grobe Stammform, um Flexionsvarianten eines Namens zu erkennen.
    w = (wort or "").strip().lower()
    for endung in KONFIDENZ_ENDUNGEN:
        if len(w) - len(endung) >= 3 and w.endswith(endung):
            return w[: -len(endung)]
    return w


def baue_konfidenzkarte(protokoll):
    if not protokoll:
        return {}

    sammler = {}
    for fenster in protokoll.get("fenster", []):
        for cluster in fenster.get("cluster", []):
            if "sicher" not in cluster:
                continue  
            name = (cluster.get("repraesentant") or "").strip().lower()
            if not name:
                continue
            gewicht = max(1, len(cluster.get("mentions", [])))
            eintrag = sammler.setdefault(
                name, {"gewicht": 0, "sicher_gewicht": 0,
                       "klassen": Counter(), "cluster": 0})
            eintrag["gewicht"] += gewicht
            eintrag["sicher_gewicht"] += gewicht * int(bool(cluster["sicher"]))
            eintrag["cluster"] += 1
            klasse = cluster.get("belegklasse")
            if klasse:
                eintrag["klassen"][klasse] += gewicht

    karte = {}
    for name, e in sammler.items():
        schwaechste = None
        if e["klassen"]:
            schwaechste = min(e["klassen"],
                              key=lambda k: BELEGKLASSEN_RANG.get(k, 99))
        karte[name] = {
            "sicher": e["sicher_gewicht"] / e["gewicht"],
            "belegklasse": schwaechste,
            "cluster": e["cluster"],
        }

    #Stammformen als zweite Ebene, damit Flexionsvarianten treffen
    nach_stamm = {}
    for name, wert in karte.items():
        nach_stamm.setdefault(konfidenz_stamm(name), []).append(wert)
    return {"exakt": karte, "stamm": nach_stamm}


def konfidenz_fuer(name, karte):
    #Sucht die Konfidenzangaben zu einem eingesetzten Figurennamen.
    leer = {"coref_verknuepft": False, "coref_sicher": None,
            "coref_belegklasse": None, "coref_cluster_anzahl": 0}
    if not karte or not name:
        return leer

    schluessel = str(name).strip().lower()
    treffer = karte["exakt"].get(schluessel)
    if treffer is not None:
        kandidaten = [treffer]
    else:
        kandidaten = karte["stamm"].get(konfidenz_stamm(schluessel), [])
    if not kandidaten:
        return leer

    anzahl = sum(k["cluster"] for k in kandidaten)
    sicher = sum(k["sicher"] * k["cluster"] for k in kandidaten) / max(1, anzahl)
    klassen = [k["belegklasse"] for k in kandidaten if k["belegklasse"]]
    schwaechste = (min(klassen, key=lambda k: BELEGKLASSEN_RANG.get(k, 99))
                   if klassen else None)
    return {"coref_verknuepft": True, "coref_sicher": round(sicher, 3),
            "coref_belegklasse": schwaechste, "coref_cluster_anzahl": anzahl}


def ergaenze_konfidenz(zuordnungen, protokoll):
    karte = baue_konfidenzkarte(protokoll)
    for z in zuordnungen:
        if z.get("methode") == "coref" and z.get("status") == "zugeordnet":
            z.update(konfidenz_fuer(z.get("figur"), karte))
        else:
            z.update({"coref_verknuepft": None, "coref_sicher": None,
                      "coref_belegklasse": None, "coref_cluster_anzahl": None})
    return zuordnungen


print("Schritt 9b bereit: Konfidenzfelder der Coreference Resolution")

Schritt 9b bereit: Konfidenzfelder der Coreference Resolution


In [ ]:
def verarbeite_maerchen(text, quelle, aliase=ALIASE, bereinigen=True):
    #Führt die komplette Pipeline für 1 Märchen aus und liefert ein Ergebnis-Dict zurück
    
    #--- Schritt 1: Normalisieren, parsen, Figuren --------------------------
    text = normalisiere_text(text)
    doc = nlp(text)

    figuren_ner, figuren_detail = extrahiere_figuren_ner_detail(doc)
    figurenindex, schluessel_info, figuren_bereinigt = baue_figurenindex(
        figuren_detail, aliase=aliase, bereinigen=bereinigen
    )

    #Lokaler Anzeigename je Schluessel
    anzeige_lokal = {
        s: waehle_anzeigenamen(s, info["formen"], info["merkmale"])
        for s, info in schluessel_info.items()
    }
    figuren_kanonisch = Counter({
        anzeige_lokal[s]: info["anzahl"] for s, info in schluessel_info.items()
    })
    figuren = set(figuren_kanonisch)

    #--- Schritt 2: Coreference Resolution ---------
    coref_ok = True
    coref_protokoll = None
    try:
        coref_text_roh, coref_protokoll = resolve_coreferences_protokolliert(text)
        coref_text = normalisiere_text(coref_text_roh)
    except Exception as fehler:
        coref_ok = False
        coref_text = text  #Fallback: ohne Coref weiterrechnen
        print(f"    ! Coref fehlgeschlagen fuer {quelle}: {fehler} -> Fallback ohne Coref")

    #--- Schritt 3: Sätze mit IDs -----------------------------------------
    satz_spans = repariere_satzspannen(doc.sents)
    saetze = [{"id": i, "text": span.text.strip()} for i, span in enumerate(satz_spans)]

    #--- Schritt 4: Satzmapping Original <-> Coref -------------------------
    doc_coref = nlp(coref_text)
    saetze_coref = [span.text.strip() for span in repariere_satzspannen(doc_coref.sents)]
    #Zuordnung über die Tokenfolge statt über Satzzahl und Satzähnlichkeit
    satz_mapping = baue_satzmapping([s["text"] for s in saetze], saetze_coref)

    #--- Schritte 5/6: Adjektive je Satz -----------------------------------
    satz_ids_mit_adj = [
        s["id"] for s in saetze
        if any(ist_adjektiv(t) for t in satz_spans[s["id"]])
    ]

    #--- Schritt 7: Adjektiv -> Figur --------------------------------------
    zuordnungen = []
    for sid in satz_ids_mit_adj:
        zuordnungen.extend(
            extrahiere_adjektiv_zuordnungen(satz_spans[sid], sid, figurenindex,
                                            methode="original")
        )

    #--- Schritte 8/9: offene Pronomen-Faelle über Coref auflösen -------
    coref_satz_von = {m["id"]: m["coref"] for m in satz_mapping}
    coref_cache = {}
    for z in zuordnungen:
        if z["status"] != "offen":
            continue
        #Gesperrte Belege nicht auflösen
        if z.get("bezug_gesperrt"):
            continue
        coref_satz = coref_satz_von.get(z["satz_id"])
        if not coref_satz:
            continue
        if z["satz_id"] not in coref_cache:
            coref_cache[z["satz_id"]] = extrahiere_adjektiv_zuordnungen(
                nlp(coref_satz), z["satz_id"], figurenindex, methode="coref"
            )
        for kandidat in coref_cache[z["satz_id"]]:
            gleiches_adjektiv = (kandidat["lemma"] == z["lemma"]
                                 or kandidat["adjektiv"].lower() == z["adjektiv"].lower())
            if kandidat["figur"] and gleiches_adjektiv:
                z.update(figur=kandidat["figur"],
                         figur_schluessel=kandidat["figur_schluessel"],
                         bezugswort=kandidat["bezugswort"],
                         bezug_pos=kandidat["bezug_pos"],
                         konstruktion=kandidat["konstruktion"],
                         numerus=kandidat["numerus"],
                         kollektiv=kandidat["kollektiv"],
                         bezug_regel=kandidat["bezug_regel"],
                         bezug_klasse=kandidat["bezug_klasse"],
                         bezug_sicher=kandidat["bezug_sicher"],
                         bezug_nominalkopf=kandidat["bezug_nominalkopf"],
                         bezug_token_i=None,
                         methode="coref", status="zugeordnet", grund_offen=None)
                break

    #--- Schritt 9b: Konfidenz der Coreference Resolution -------------
    ergaenze_konfidenz(zuordnungen, coref_protokoll)

    #--- Ergebnis: Figurenprofil dieses Märchens --------------------------
    #Schlüssel als Profilschlüssel; Anzeigename folgt nach dem Batch
    figuren_profil = defaultdict(list)
    for z in zuordnungen:
        if z["status"] == "zugeordnet":
            figuren_profil[z["figur_schluessel"]].append({
                "adjektiv": z["adjektiv"],
                "lemma": z["lemma"],
                "satz_id": z["satz_id"],
                "methode": z["methode"],
                "numerus": z["numerus"],
                "kollektiv": z["kollektiv"],
                "coref_sicher": z.get("coref_sicher"),
                "coref_belegklasse": z.get("coref_belegklasse"),
                "bezug_klasse": z.get("bezug_klasse"),
                "bezug_sicher": z.get("bezug_sicher"),
            })


    #--- Schritt 10: Instanzebene ---------------------
    _zerfall = globals().get("zerfall_rollen_je_maerchen", {}).get(quelle)
    instanz_ergebnis = inst.verarbeite_instanzebene(
        satz_docs=[(s["id"], satz_spans[s["id"]]) for s in saetze],
        figurenindex=figurenindex,
        zuordnungen=zuordnungen,
        quelle=quelle,
        zerfall_rollen=_zerfall,
        lemmafeld="lemma_norm" if any("lemma_norm" in z for z in zuordnungen)
                  else "lemma",
    )

    statistik = {
        "saetze": len(saetze),
        "saetze_mit_adjektiv": len(satz_ids_mit_adj),
        "adjektive_gesamt": len(zuordnungen),
        "adjektive_zugeordnet": sum(1 for z in zuordnungen if z["status"] == "zugeordnet"),
        "adjektive_offen": sum(1 for z in zuordnungen if z["status"] == "offen"),
        "adjektive_via_coref": sum(1 for z in zuordnungen if z["methode"] == "coref"),
        "figuren": len(figuren),
        "adjektive_kollektiv": sum(1 for z in zuordnungen
                                   if z["status"] == "zugeordnet" and z["kollektiv"]),
        "ner_rohformen": len(figuren_ner),
        "ner_bereinigt": len(figuren_bereinigt),
        "adjektive_coref_unsicher": sum(
            1 for z in zuordnungen
            if z.get("coref_sicher") is not None and z["coref_sicher"] < 1.0),
        "adjektive_coref_ohne_cluster": sum(
            1 for z in zuordnungen if z.get("coref_verknuepft") is False),
        "adjektive_bezug_unsicher": sum(
            1 for z in zuordnungen
            if z["status"] == "zugeordnet" and z.get("bezug_sicher") is False),
        "adjektive_bezug_adverbial": sum(
            1 for z in zuordnungen
            if z.get("bezug_klasse") == "adverbial_ohne_figurenbezug"),
        "adjektive_gesperrt": sum(1 for z in zuordnungen
                                  if z.get("bezug_gesperrt")),
        "erwaehnungen": len(instanz_ergebnis["erwaehnungen"]),
        "instanzen": len({e["instanz"] for e in instanz_ergebnis["erwaehnungen"]
                          if e["instanz"]}),
        "adjektive_instanz_zugewiesen":
            instanz_ergebnis["abdeckung"]["instanz_zugewiesen"],
    }

    return {
        "quelle": quelle,
        "erwaehnungen": instanz_ergebnis["erwaehnungen"],
        "instanzprofile": instanz_ergebnis["instanzprofile"],
        "instanz_abdeckung": instanz_ergebnis["abdeckung"],
        "instanz_plausibilitaet": instanz_ergebnis["plausibilitaet"],
        "instanz_protokoll": instanz_ergebnis["instanz_protokoll"],
        "instanz_verdacht_befunde": instanz_ergebnis["verdacht_befunde"],
        "figuren": sorted(figuren),
        "figuren_ner": dict(figuren_ner),
        "figuren_kanonisch": dict(figuren_kanonisch),
        "satz_mapping": satz_mapping,
        "zuordnungen": zuordnungen,
        "figuren_profil": dict(figuren_profil),
        "statistik": statistik,
        "coref_ok": coref_ok,
        "figuren_ner_detail": figuren_detail,
        "figuren_bereinigt": figuren_bereinigt,
        "figuren_schluessel": sorted(schluessel_info),
        "anzeige_lokal": anzeige_lokal,
        "schluessel_formen": {s: dict(i["formen"]) for s, i in schluessel_info.items()},
        "schluessel_merkmale": {s: i["merkmale"] for s, i in schluessel_info.items()},
        "schluessel_numerus": {s: dict(i["numerus"]) for s, i in schluessel_info.items()},
        "mapping_luecken": sum(1 for m in satz_mapping if m["coref_id"] is None),
        "mapping_verworfen": sum(1 for m in satz_mapping if m["verworfen"]),
        "coref_protokoll": coref_protokoll,
    }

## Alle Märchen verarbeiten (Batch)

In [ ]:
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kw):
        return x

PROTOKOLL_ORDNER = Path("protokolle")
PROTOKOLL_ORDNER.mkdir(exist_ok=True)

ergebnisse_pro_maerchen = []   # ein verarbeite_maerchen()-Ergebnis je Märchen
fehler_maerchen = []           # Märchen, die nicht durchliefen (mit Grund)

for m in tqdm(maerchen_dateien, desc="Märchen"):
    try:
        text = m["pfad"].read_text(encoding="utf-8")
    except Exception as fehler:
        fehler_maerchen.append({"quelle": m["quelle"], "fehler": f"Lesefehler: {fehler}"})
        print(f"  FEHLER {m['titel']}: konnte nicht gelesen werden ({fehler})")
        continue

    try:
        res = verarbeite_maerchen(text, m["quelle"])
    except Exception as fehler:
        fehler_maerchen.append({"quelle": m["quelle"], "fehler": repr(fehler)})
        print(f"  FEHLER {m['titel']}: Pipeline-Fehler ({fehler})")
        continue

    res["sammlung"] = m["sammlung"]
    res["titel"] = m["titel"]

    protokoll = res.pop("coref_protokoll", None)
    if protokoll is not None:
        protokoll_datei = f"protokoll_{m['titel']}.json"
        (PROTOKOLL_ORDNER / protokoll_datei).write_text(
            json.dumps(protokoll, ensure_ascii=False), encoding="utf-8")
        res["coref_protokoll_datei"] = protokoll_datei
    ergebnisse_pro_maerchen.append(res)

    st = res["statistik"]
    print(f"  Märchen verarbeitet:  {m['titel']:<28} Figuren={st['figuren']:<3} "
          f"Adj. zugeordnet={st['adjektive_zugeordnet']:<3} Adj. offen={st['adjektive_offen']:<3} "
          f"(davon Adj. via coref: +{st['adjektive_via_coref']})")

print(f"\nFertig: {len(ergebnisse_pro_maerchen)} Märchen verarbeitet, "
      f"{len(fehler_maerchen)} fehlgeschlagen.")
if fehler_maerchen:
    print("Fehlgeschlagene Märchen:")
    for f in fehler_maerchen:
        print(f"  - {f['quelle']}: {f['fehler']}")

Märchen:   0%|          | 0/80 [00:00<?, ?it/s]

--- Verarbeite Fenster 1/1 ---


Märchen:   1%|▏         | 1/80 [00:00<00:30,  2.57it/s]

  Märchen verarbeitet:  1001 Nacht Epilog            Figuren=16  Adj. zugeordnet=4   Adj. offen=16  (davon Adj. via coref: +1)
--- Verarbeite Fenster 1/5 ---
--- Verarbeite Fenster 2/5 ---
--- Verarbeite Fenster 3/5 ---
--- Verarbeite Fenster 4/5 ---
--- Verarbeite Fenster 5/5 ---


Märchen:   2%|▎         | 2/80 [00:03<02:14,  1.73s/it]

...

  Märchen verarbeitet:  Rai-taro                     Figuren=29  Adj. zugeordnet=32  Adj. offen=93  (davon Adj. via coref: +6)
--- Verarbeite Fenster 1/6 ---
--- Verarbeite Fenster 2/6 ---
--- Verarbeite Fenster 3/6 ---
--- Verarbeite Fenster 4/6 ---
--- Verarbeite Fenster 5/6 ---
--- Verarbeite Fenster 6/6 ---


Märchen:  99%|█████████▉| 79/80 [02:59<00:01,  1.78s/it]

  Märchen verarbeitet:  Urashima Taro                Figuren=47  Adj. zugeordnet=104 Adj. offen=175 (davon Adj. via coref: +43)
--- Verarbeite Fenster 1/2 ---
--- Verarbeite Fenster 2/2 ---


Märchen: 100%|██████████| 80/80 [03:00<00:00,  2.26s/it]

  Märchen verarbeitet:  Wenn man mit Kobolden tanzt  Figuren=8   Adj. zugeordnet=17  Adj. offen=27  (davon Adj. via coref: +11)

Fertig: 80 Märchen verarbeitet, 0 fehlgeschlagen.


## Korpusweiter Nachlauf

In [ ]:
DATIVENDUNGEN = ("e", "n", "en")

def konsolidiere_dativformen(formen_je_schluessel):
    belegt = set(formen_je_schluessel)
    ersetzung, protokoll = {}, []

    for schluessel in sorted(belegt, key=lambda s: (-len(s), s)):
        if " " in schluessel:
            continue
        for endung in DATIVENDUNGEN:
            if not schluessel.endswith(endung):
                continue
            ziel = schluessel[: -len(endung)]
            if len(ziel) < 3 or ziel not in belegt:
                continue
            while ziel in ersetzung:      #Ketten aufloesen
                ziel = ersetzung[ziel]
            if ziel == schluessel:
                continue
            ersetzung[schluessel] = ziel
            protokoll.append({
                "von": schluessel, "nach": ziel, "endung": endung,
                "nennungen_von": sum(formen_je_schluessel[schluessel].values()),
                "nennungen_nach": sum(formen_je_schluessel[ziel].values()),
            })
            break
    return ersetzung, protokoll


def vereinheitliche_figurennamen(ergebnisse, aliase=ALIASE,
                                 schranke=True, konsolidieren=True):
    #--- 1) Alle Oberflächenformen und Merkmale je Schlüssel sammeln -----
    formen_je_schluessel = defaultdict(Counter)
    merkmale_je_schluessel = defaultdict(list)

    for res in ergebnisse:
        for schluessel, formen in res.get("schluessel_formen", {}).items():
            for form, n in formen.items():
                formen_je_schluessel[schluessel][form] += n
        for schluessel, merkmale in res.get("schluessel_merkmale", {}).items():
            merkmale_je_schluessel[schluessel].extend(merkmale)

    #--- 2) Plausibilitaetsschranke auf Basis der korpusweiten Evidenz -----
    evidenz = baue_ner_evidenz(ergebnisse)
    verworfene_schluessel, schranken_protokoll = set(), []
    kuerzungen = {}

    if schranke:
        for schluessel, merkmale_liste in list(merkmale_je_schluessel.items()):
            if " " not in schluessel:
                continue
            #Repräsentant: häufigste Rohform dieses Schlüssels
            haeufigste = formen_je_schluessel[schluessel].most_common(1)[0][0]
            vertreter = next((m for m in merkmale_liste
                              if m["rohform"] == haeufigste), merkmale_liste[0])

            entscheidung, ziel, grund = pruefe_mehrwortname(vertreter, evidenz)
            if entscheidung == "behalten":
                continue

            eintrag = {"schluessel": schluessel, "rohform": haeufigste,
                       "entscheidung": entscheidung, "ziel": ziel, "grund": grund,
                       "nennungen": sum(formen_je_schluessel[schluessel].values())}
            schranken_protokoll.append(eintrag)

            if entscheidung == "verwerfen":
                verworfene_schluessel.add(schluessel)
            elif entscheidung == "kuerzen" and ziel:
                zielschluessel = _normform(ziel)
                if zielschluessel in formen_je_schluessel:
                    kuerzungen[schluessel] = zielschluessel

    #--- 3) Verworfene und gekürzte Schlüssel ausbuchen ------------------
    for schluessel in verworfene_schluessel:
        formen_je_schluessel.pop(schluessel, None)
        merkmale_je_schluessel.pop(schluessel, None)
    for von, nach in kuerzungen.items():
        formen_je_schluessel[nach].update(formen_je_schluessel.pop(von))
        merkmale_je_schluessel[nach].extend(merkmale_je_schluessel.pop(von, []))

    #--- 4) Dativformen konsolidieren --------------------------------------
    ersetzung, konsolidierungsprotokoll = (
        konsolidiere_dativformen(formen_je_schluessel) if konsolidieren else ({}, [])
    )
    ersetzung.update(kuerzungen)
    for von, nach in list(ersetzung.items()):
        if von in formen_je_schluessel and von != nach:
            formen_je_schluessel[nach].update(formen_je_schluessel.pop(von))
            merkmale_je_schluessel[nach].extend(merkmale_je_schluessel.pop(von, []))

    #--- 5) Je Schluessel einen Anzeigenamen bestimmen ---------------------
    #Alias-Namen, falls sie wieder aufgenommen werden, hätten Vorrang
    alias_namen = {v.strip().lower(): v for v in (aliase or {}).values()} 

    anzeigename_von = {}
    for schluessel, formen in formen_je_schluessel.items():
        if schluessel in alias_namen:
            anzeigename_von[schluessel] = alias_namen[schluessel]
        else:
            anzeigename_von[schluessel] = waehle_anzeigenamen(
                schluessel, formen, merkmale_je_schluessel.get(schluessel))

    #--- 6) Namenskollisionen auflösen ------------------------------------
    belegt = {}
    kollisionsprotokoll = []
    for schluessel in sorted(anzeigename_von,
                             key=lambda s: -sum(formen_je_schluessel[s].values())):
        name = anzeigename_von[schluessel]
        if name not in belegt:
            belegt[name] = schluessel
            continue

        sieger = belegt[name]
        gemeinsam = set(formen_je_schluessel[schluessel]) & set(formen_je_schluessel[sieger])
        if gemeinsam:
            #Dasselbe Wort in zwei Lemmatisierungsvarianten -> zusammenführen
            ersetzung[schluessel] = sieger
            formen_je_schluessel[sieger].update(formen_je_schluessel.pop(schluessel))
            merkmale_je_schluessel[sieger].extend(merkmale_je_schluessel.pop(schluessel, []))
            anzeigename_von.pop(schluessel, None)
            kollisionsprotokoll.append({
                "von": schluessel, "nach": sieger, "anzeigename": name,
                "gemeinsame_formen": sorted(gemeinsam), "entscheidung": "zusammengefuehrt",
            })
        else:
            #Echte Namensgleichheit verschiedener Figuren -> disambiguieren
            anzeigename_von[schluessel] = f"{name} [{schluessel}]"
            kollisionsprotokoll.append({
                "von": schluessel, "nach": None, "anzeigename": name,
                "gemeinsame_formen": [], "entscheidung": "disambiguiert",
            })

    #--- 7) Alles in die Ergebnisse zurückschreiben -----------------------
    def ziel_von(schluessel):
        """Folgt der Ersetzungskette bis zum endgueltigen Schluessel."""
        gesehen = set()
        while schluessel in ersetzung and schluessel not in gesehen:
            gesehen.add(schluessel)
            schluessel = ersetzung[schluessel]
        return schluessel

    for res in ergebnisse:
        for z in res["zuordnungen"]:
            s = z.get("figur_schluessel")
            if not s:
                continue
            if s in verworfene_schluessel:
                #Zuordnung zurücknehmen, aber sichtbar lassen
                z.update(figur=None, figur_schluessel=None,
                         status="offen", grund_offen="figur_verworfen")
                continue
            s = ziel_von(s)
            z["figur_schluessel"] = s
            z["figur"] = anzeigename_von.get(s)

        profil_neu = defaultdict(list)
        for schluessel, eintraege in res["figuren_profil"].items():
            if schluessel in verworfene_schluessel:
                continue
            ziel = ziel_von(schluessel)
            profil_neu[anzeigename_von.get(ziel, ziel)].extend(eintraege)
        res["figuren_profil"] = dict(profil_neu)

        kanonisch_neu = Counter()
        for schluessel, formen in res.get("schluessel_formen", {}).items():
            if schluessel in verworfene_schluessel:
                continue
            ziel = ziel_von(schluessel)
            kanonisch_neu[anzeigename_von.get(ziel, ziel)] += sum(formen.values())
        res["figuren_kanonisch"] = dict(kanonisch_neu)
        res["figuren"] = sorted(kanonisch_neu)

        #Statistik nachziehen: die Schranke verändert die Zählungen
        z_alle = res["zuordnungen"]
        res["statistik"].update(
            figuren=len(kanonisch_neu),
            adjektive_zugeordnet=sum(1 for z in z_alle if z["status"] == "zugeordnet"),
            adjektive_offen=sum(1 for z in z_alle if z["status"] == "offen"),
            adjektive_via_coref=sum(1 for z in z_alle
                                    if z["status"] == "zugeordnet" and z["methode"] == "coref"),
        )

    protokoll = {
        "schranke": schranken_protokoll,
        "konsolidierung": konsolidierungsprotokoll,
        "kollisionen": kollisionsprotokoll,
        "verworfene_schluessel": sorted(verworfene_schluessel),
        "evidenz_schwelle": MIN_EVIDENZ_FIGURENNOMEN,
    }
    return anzeigename_von, formen_je_schluessel, protokoll


anzeigename_von, formen_je_schluessel, nachlauf_protokoll = (
    vereinheitliche_figurennamen(ergebnisse_pro_maerchen)
)
konsolidierungsprotokoll = nachlauf_protokoll["konsolidierung"]
schranken_protokoll = nachlauf_protokoll["schranke"]
kollisionsprotokoll = nachlauf_protokoll["kollisionen"]

print(f"{len(anzeigename_von)} Schlüssel korpusweit -> Anzeigenamen vergeben\n")

#--- Zwischenausgabe 1: was die Schranke entschieden hat --------------------
verworfen = [e for e in schranken_protokoll if e["entscheidung"] == "verwerfen"]
gekuerzt = [e for e in schranken_protokoll if e["entscheidung"] == "kuerzen"]
print(f"=== Plausibilitätsschranke (Schwelle {MIN_EVIDENZ_FIGURENNOMEN}) ===")
print(f"verworfen: {len(verworfen)}   gekürzt: {len(gekuerzt)}\n")
for e in sorted(verworfen, key=lambda x: -x["nennungen"])[:20]:
    print(f"  verworfen  {e['rohform']:<32} {e['grund']}")
for e in sorted(gekuerzt, key=lambda x: -x["nennungen"])[:10]:
    print(f"  gekürzt    {e['rohform']:<32} -> {e['ziel']}")
print()

#--- Zwischenausgabe 2: konsolidierte Dativformen --------------------------
print(f"=== Über Dativendungen zusammengeführt: {len(konsolidierungsprotokoll)} ===")
for e in sorted(konsolidierungsprotokoll, key=lambda x: -x["nennungen_von"])[:15]:
    print(f"  {e['von']:<18} -> {e['nach']:<18} "
          f"(-{e['endung']}, {e['nennungen_von']} + {e['nennungen_nach']} Nennungen)")
print()

#--- Zwischenausgabe 3: was der Nachlauf zusammengeführt hat ---------------
zusammengefuehrt = sorted(
    ((s, f) for s, f in formen_je_schluessel.items() if len(f) > 1),
    key=lambda x: -sum(x[1].values()))
print(f"=== Schlüssel mit mehr als einer Oberflächenform: {len(zusammengefuehrt)} ===")
for schluessel, formen in zusammengefuehrt[:20]:
    varianten = ", ".join(f"{f}×{n}" for f, n in formen.most_common(6))
    print(f"  {anzeigename_von[schluessel]:<22} <- {varianten}")


#--- Zwischenausgabe 4: Namenskollisionen ----------------------------------
zusammen = [e for e in kollisionsprotokoll if e["entscheidung"] == "zusammengefuehrt"]
disamb = [e for e in kollisionsprotokoll if e["entscheidung"] == "disambiguiert"]
print()
print(f"=== Namenskollisionen: {len(zusammen)} zusammengeführt, "
      f"{len(disamb)} disambiguiert ===")
for e in zusammen[:12]:
    print(f"  zusammengeführt  '{e['anzeigename']}': {e['von']} -> {e['nach']} "
          f"(gemeinsame Formen: {', '.join(e['gemeinsame_formen'][:3])})")
for e in disamb[:8]:
    print(f"  disambiguiert    '{e['anzeigename']}' -> '{e['anzeigename']} [{e['von']}]'")

1090 Schlüssel korpusweit -> Anzeigenamen vergeben

=== Plausibilitätsschranke (Schwelle 4) ===
verworfen: 1   gekürzt: 6

  verworfen  Buddhas Bote                     genitiv_sachnomen (Kopf 2x belegt)
  gekürzt    Pari Banu                        -> Banu
  gekürzt    Nachlässiger Fremdling           -> Fremdling
  gekürzt    Rief Chodadad                    -> Chodadad
  gekürzt    Rief Ali Baba                    -> Ali Baba
  gekürzt    Dummer Teufel                    -> Teufel
  gekürzt    Japaner Wasobiowo                -> Wasobiowo

=== Über Dativendungen zusammengeführt: 71 ===
  sklave             -> sklav              (-e, 40 + 3 Nennungen)
  alte               -> alt                (-e, 40 + 31 Nennungen)
  manne              -> mann               (-e, 24 + 268 Nennungen)
  junge              -> jung               (-e, 21 + 3 Nennungen)
  däumeliesche       -> däumeliesch        (-e, 14 + 35 Nennungen)
  brüdern            -> brüder             (-n, 11 + 1 Nennungen)
  so

### Schritt 10c — Belebtheitskonfidenz der Figurenschlüssel

In [ ]:
BELEBT_SCHWELLEN = {
    "anker_min_propn_anteil": 0.80,   #Anteil PROPN-Taggings der Vorkommen
    "anker_min_belege": 10,           #Mindestzahl Vorkommen im Korpus
    "verb_min_belege": 10,            #Mindestbesetzung je Verb
    "verb_min_anker": 5,              #Mindestzahl Ankertreffer je Verb
    "verb_min_lift": 1.50,            #Faktor gegenueber der Grundrate
    "agens_min_treffer": 1,
    "adressat_min_treffer": 1,
    "rede_min_treffer": 1,
    "koref_min_cluster": 1,
    "sicher_ab_signalen": 1,          #ab wie vielen Signalen gilt 'bestaetigt'
}

#Stellen des TIGER-Schemas. Grammatische Kategorien, keine inhaltliche Auswahl
SLOT_SUBJEKT, SLOT_DATIV, SLOT_VOKATIV = "sb", "da", "vo"
SLOT_ZU, SLOT_REDE = "zu_op", "rede_sb"
SLOTS_ARGUMENT = (SLOT_SUBJEKT, SLOT_DATIV, SLOT_ZU)

#Anführungszeichen eines Satzes
ZITATZEICHEN = set('»«„“”‚‘›‹"\'')

#Geschlossene grammatische Klasse der Personalpronomen der dritten Person
PERSONALPRONOMEN_3P = {"er", "sie", "es", "ihn", "ihm", "ihr", "ihnen",
                       "sich", "derselbe", "dieselbe", "dasselbe"}


def _belebt_normform(s):
    return (str(s).strip().lower() if s is not None else "")


def zitatmaske(doc):
    #Markiert je Token, ob es innerhalb einer Redepassage steht.
    innen, offen = [False] * len(doc), False
    for i, tok in enumerate(doc):
        if tok.text in ZITATZEICHEN:
            innen[i] = True
            offen = not offen
            continue
        innen[i] = offen
    return innen


def fuehrt_rede_ein(verb, innen, doc, fenster=5):
    #Prueft, ob ein Verb direkte Rede einfuehrt
    if innen[verb.i]:
        return False
    satz = verb.sent
    if not any(t.text in ZITATZEICHEN for t in satz):
        return False
    von, bis = max(satz.start, verb.i - fenster), min(satz.end, verb.i + fenster + 1)
    return any(doc[j].text in ZITATZEICHEN or doc[j].text == ":"
               for j in range(von, bis))


def _bestimme_slot(tok):
    #Ordnet ein Token einer der erhobenen Argumentstellen zu
    kopf = tok.head
    if tok.dep_ == SLOT_SUBJEKT and kopf.pos_ in {"VERB", "AUX"}:
        return SLOT_SUBJEKT, kopf.lemma_.lower()
    if tok.dep_ == SLOT_DATIV and kopf.pos_ in {"VERB", "AUX"}:
        return SLOT_DATIV, kopf.lemma_.lower()
    if tok.dep_ == SLOT_VOKATIV:
        return SLOT_VOKATIV, (kopf.lemma_.lower()
                              if kopf.pos_ in {"VERB", "AUX"} else None)
    if kopf.pos_ == "ADP" and _belebt_normform(kopf.lemma_) in {"zu", "zum", "zur"}:
        gross = kopf.head
        if gross.pos_ in {"VERB", "AUX"}:
            return SLOT_ZU, gross.lemma_.lower()
    return None, None


def baue_form_index(formen_je_schluessel):
    index = {}
    for schluessel, formen in sorted(formen_je_schluessel.items()):
        gesamt = sum(formen.values())
        for form in list(formen) + [schluessel]:
            f = _belebt_normform(form)
            if not f:
                continue
            vorher = index.get(f)
            if vorher is None or sum(formen_je_schluessel.get(vorher, {}).values()) < gesamt:
                index[f] = schluessel
    return index


def sammle_syntaxevidenz(dateien, form_index, ner_maerchen_je_schluessel):
    slot_verb_schluessel = defaultdict(lambda: defaultdict(Counter))
    slot_schluessel_verb = defaultdict(lambda: defaultdict(Counter))
    slot_schluessel_verb_lokal = defaultdict(lambda: defaultdict(Counter))
    pos_je_schluessel = defaultdict(Counter)

    aus = [name for name in ("ner", "quote_grenzen")
           if name in nlp.pipe_names]
    try:
        from tqdm.auto import tqdm
        lauf = tqdm(dateien, desc="Belebtheitsevidenz")
    except ImportError:
        lauf = dateien

    for eintrag in lauf:
        text = normalisiere_text(Path(eintrag["pfad"]).read_text(encoding="utf-8"))
        quelle = eintrag["quelle"]
        with nlp.select_pipes(disable=aus):
            doc = nlp(text)
        innen = zitatmaske(doc)

        for tok in doc:
            if tok.pos_ not in {"NOUN", "PROPN"}:
                continue
            schluessel = (form_index.get(_belebt_normform(tok.lemma_))
                          or form_index.get(_belebt_normform(tok.text)))
            if schluessel:
                pos_je_schluessel[schluessel][tok.pos_] += 1

            if (tok.dep_ == SLOT_SUBJEKT and tok.head.pos_ in {"VERB", "AUX"}
                    and fuehrt_rede_ein(tok.head, innen, doc)):
                redeverb = tok.head.lemma_.lower()
                slot_verb_schluessel[SLOT_REDE][redeverb][
                    schluessel or "(kein_schluessel)"] += 1
                if schluessel:
                    slot_schluessel_verb[SLOT_REDE][schluessel][redeverb] += 1
                    if quelle in ner_maerchen_je_schluessel.get(schluessel, set()):
                        slot_schluessel_verb_lokal[SLOT_REDE][schluessel][redeverb] += 1

            slot, verb = _bestimme_slot(tok)
            if slot is None:
                continue
            slot_verb_schluessel[slot][verb][schluessel or "(kein_schluessel)"] += 1
            if schluessel:
                slot_schluessel_verb[slot][schluessel][verb] += 1
                if quelle in ner_maerchen_je_schluessel.get(schluessel, set()):
                    slot_schluessel_verb_lokal[slot][schluessel][verb] += 1

    return (slot_verb_schluessel, slot_schluessel_verb,
            slot_schluessel_verb_lokal, pos_je_schluessel)


def bestimme_ankermenge(pos_je_schluessel):
    anker = set()
    for schluessel, verteilung in pos_je_schluessel.items():
        gesamt = sum(verteilung.values())
        if gesamt < BELEBT_SCHWELLEN["anker_min_belege"]:
            continue
        if (verteilung.get("PROPN", 0) / gesamt
                >= BELEBT_SCHWELLEN["anker_min_propn_anteil"]):
            anker.add(schluessel)
    return anker


def bilde_verbklasse(slot_verb_schluessel, anker):
    zaehler = defaultdict(Counter)
    for slot in SLOTS_ARGUMENT:
        for verb, treffer in slot_verb_schluessel.get(slot, {}).items():
            if verb is not None:
                zaehler[verb].update(treffer)

    anker_gesamt = sum(n for c in zaehler.values()
                       for s, n in c.items() if s in anker)
    alle_gesamt = sum(sum(c.values()) for c in zaehler.values())
    grundrate = anker_gesamt / alle_gesamt if alle_gesamt else 0.0

    klasse, tabelle = set(), []
    for verb, treffer in zaehler.items():
        belege = sum(treffer.values())
        anker_treffer = sum(n for s, n in treffer.items() if s in anker)
        if belege < BELEBT_SCHWELLEN["verb_min_belege"]:
            continue
        anteil = anker_treffer / belege
        lift = anteil / grundrate if grundrate else 0.0
        drin = (lift >= BELEBT_SCHWELLEN["verb_min_lift"]
                and anker_treffer >= BELEBT_SCHWELLEN["verb_min_anker"])
        tabelle.append({"verb": verb, "belege": belege,
                        "anker_treffer": anker_treffer, "anteil": anteil,
                        "lift": lift, "in_klasse": drin})
        if drin:
            klasse.add(verb)
    tabelle.sort(key=lambda z: (-z["in_klasse"], -z["lift"]))
    return klasse, grundrate, tabelle


_pron_cache = {}


def _ist_personalpronomen(text):
    #Wortart einer Erwaehnung, bevorzugt am Modell bestimmt
    s = _belebt_normform(text)
    if s in _pron_cache:
        return _pron_cache[s]
    treffer = False
    if s and len(s.split()) == 1:
        tok = nlp(s)[0]
        treffer = (tok.pos_ == "PRON"
                   and "Prs" in tok.morph.get("PronType")
                   and "3" in tok.morph.get("Person"))
    if not treffer:
        treffer = s in PERSONALPRONOMEN_3P
    _pron_cache[s] = treffer
    return treffer


def sammle_koreferenzevidenz(ergebnisse, form_index, ordner=PROTOKOLL_ORDNER):
    rep, weit, gelesen = Counter(), Counter(), 0

    def schluessel_der(text):
        gefunden = set()
        ganz = form_index.get(_belebt_normform(text))
        if ganz:
            gefunden.add(ganz)
        for w in re.findall(r"[\wÄÖÜäöüß]+", str(text or "")):
            s = form_index.get(_belebt_normform(w))
            if s:
                gefunden.add(s)
        return gefunden

    for res in ergebnisse:
        datei = res.get("coref_protokoll_datei")
        if not datei:
            continue
        pfad = Path(ordner) / Path(datei).name
        if not pfad.is_file():
            continue
        with open(pfad, encoding="utf-8") as f:
            protokoll = json.load(f)
        gelesen += 1
        for fenster in protokoll.get("fenster", []):
            for cluster in fenster.get("cluster", []):
                texte = [m.get("text", "") for m in cluster.get("mentions", [])]
                if not any(_ist_personalpronomen(t) for t in texte):
                    continue
                for s in schluessel_der(cluster.get("repraesentant") or ""):
                    rep[s] += 1
                alle = set()
                for t in texte:
                    alle |= schluessel_der(t)
                for s in alle:
                    weit[s] += 1
    return rep, weit, gelesen


def bewerte_belebtheit(schluessel_menge, koref_rep, slot_schluessel_verb,
                       verbklasse):
    
    def treffer_klasse(slot, s):
        zaehler = slot_schluessel_verb.get(slot, {}).get(s, Counter())
        return sum(n for verb, n in zaehler.items() if verb in verbklasse)

    def treffer_roh(slot, s):
        return sum(slot_schluessel_verb.get(slot, {}).get(s, Counter()).values())

    bewertung = {}
    for s in schluessel_menge:
        agens = treffer_klasse(SLOT_SUBJEKT, s)
        adressat = (treffer_roh(SLOT_DATIV, s) + treffer_roh(SLOT_VOKATIV, s)
                    + treffer_klasse(SLOT_ZU, s))
        rede = treffer_roh(SLOT_REDE, s)
        koref = koref_rep.get(s, 0)

        signale = {
            "koref": koref >= BELEBT_SCHWELLEN["koref_min_cluster"],
            "rede": rede >= BELEBT_SCHWELLEN["rede_min_treffer"],
            "agens": agens >= BELEBT_SCHWELLEN["agens_min_treffer"],
            "adressat": adressat >= BELEBT_SCHWELLEN["adressat_min_treffer"],
        }
        anzahl = sum(signale.values())
        bewertung[s] = {
            "signale": signale,
            "anzahl_signale": anzahl,
            "sicher": anzahl >= BELEBT_SCHWELLEN["sicher_ab_signalen"],
            "grund": (",".join(k for k, v in signale.items() if v)
                      or "ohne_evidenz"),
            "treffer": {"koref_cluster": koref, "rede": rede,
                        "agens": agens, "adressat": adressat},
            "einteilig": " " not in s.strip(),
        }
    return bewertung


def schreibe_belebtheitsfelder(ergebnisse, bewertung):
    gesetzt = 0
    for res in ergebnisse:
        for z in res["zuordnungen"]:
            s = z.get("figur_schluessel")
            eintrag = bewertung.get(s) if s else None
            if eintrag is None:
                z["figur_belebt_sicher"] = None
                z["figur_belebt_signale"] = None
                z["figur_belebt_grund"] = None
                continue
            z["figur_belebt_sicher"] = bool(eintrag["sicher"])
            z["figur_belebt_signale"] = int(eintrag["anzahl_signale"])
            z["figur_belebt_grund"] = eintrag["grund"]
            gesetzt += 1
    return gesetzt


#===========================================================================
# Ausführung
#===========================================================================
#In welchen Maerchen ist ein Schlüssel überhaupt als FIGUR ausgezeichnet?
ner_maerchen_je_schluessel = defaultdict(set)
for res in ergebnisse_pro_maerchen:
    for schluessel in (res.get("schluessel_formen") or {}):
        ner_maerchen_je_schluessel[schluessel].add(res["quelle"])

belebt_form_index = baue_form_index(formen_je_schluessel)

(slot_verb_schluessel, slot_schluessel_verb,
 slot_schluessel_verb_lokal, pos_je_schluessel) = sammle_syntaxevidenz(
    maerchen_dateien, belebt_form_index, ner_maerchen_je_schluessel)

ankermenge = bestimme_ankermenge(pos_je_schluessel)
verbklasse, verb_grundrate, verbtabelle = bilde_verbklasse(
    slot_verb_schluessel, ankermenge)

koref_rep, koref_weit, protokolle_gelesen = sammle_koreferenzevidenz(
    ergebnisse_pro_maerchen, belebt_form_index)

belebtheit = bewerte_belebtheit(set(formen_je_schluessel), koref_rep,
                                slot_schluessel_verb, verbklasse)
zeilen_gesetzt = schreibe_belebtheitsfelder(ergebnisse_pro_maerchen, belebtheit)

#--- Zwischenausgabe 1: Herleitung der Verbklasse --------------------------
redeverben = Counter()
for verb, treffer in slot_verb_schluessel[SLOT_REDE].items():
    redeverben[verb] += sum(treffer.values())

print(f"=== Schritt 10c — Belebtheitskonfidenz (AP1c) ===\n")
print(f"Protokolle gelesen : {protokolle_gelesen}")
print(f"Ankermenge         : {len(ankermenge)} Schlüssel "
      f"(≥ {BELEBT_SCHWELLEN['anker_min_propn_anteil']:.0%} PROPN, "
      f"≥ {BELEBT_SCHWELLEN['anker_min_belege']} Vorkommen)")
print(f"Grundrate          : {verb_grundrate:.3f}")
print(f"Verbklasse         : {len(verbklasse)} von {len(verbtabelle)} geprüften")
print("  " + ", ".join(sorted(verbklasse)))
print(f"\nRedeeinführende Verben aus der Zeichensetzung: {len(redeverben)}")
print("  " + ", ".join(f"{v} ({n})" for v, n in redeverben.most_common(10)))

#--- Zwischenausgabe 2: Verteilung der Konfidenz --------------------------
verteilung = Counter(e["anzahl_signale"] for e in belebtheit.values())
print(f"\n=== Stützung der {len(belebtheit)} Schlüssel ===")
for anzahl in sorted(verteilung):
    print(f"  {anzahl} Signal(e): {verteilung[anzahl]:>5} Schlüssel")
unbestaetigt = [s for s, e in belebtheit.items() if not e["sicher"]]
print(f"\nAls unbestätigt gekennzeichnet: {len(unbestaetigt)} Schlüssel")
print(f"Zuordnungszeilen mit Konfidenzfeld: {zeilen_gesetzt}")

#--- Zwischenausgabe 3: unbestätigte Schlüssel mit den meisten Belegen -----
belege_je_schluessel = Counter()
for res in ergebnisse_pro_maerchen:
    for z in res["zuordnungen"]:
        if z["status"] == "zugeordnet" and z.get("figur_schluessel"):
            belege_je_schluessel[z["figur_schluessel"]] += 1

print("\n=== Unbestätigte Schlüssel mit den meisten Adjektivbelegen ===")
for s in sorted(unbestaetigt, key=lambda x: -belege_je_schluessel.get(x, 0))[:20]:
    n = belege_je_schluessel.get(s, 0)
    if n == 0:
        break
    print(f"  {anzeigename_von.get(s, s):<28} {n:>3} Belege")

Belebtheitsevidenz: 100%|██████████| 80/80 [00:08<00:00,  9.80it/s]


=== Schritt 10c — Belebtheitskonfidenz (AP1c) ===

Protokolle gelesen : 80
Ankermenge         : 30 Schlüssel (≥ 80% PROPN, ≥ 10 Vorkommen)
Grundrate          : 0.044
Verbklasse         : 7 von 117 geprüften
  erblicken, erzählen, fragen, rufen, sagen, sehen, weinen

Redeeinführende Verben aus der Zeichensetzung: 178
  sagen (296), sprechen (170), antworten (123), rufen (62), erwidern (49), sein (48), fragen (47), entgegnen (23), denken (22), haben (16)

=== Stützung der 1090 Schlüssel ===
  0 Signal(e):   653 Schlüssel
  1 Signal(e):   196 Schlüssel
  2 Signal(e):    80 Schlüssel
  3 Signal(e):    73 Schlüssel
  4 Signal(e):    88 Schlüssel

Als unbestätigt gekennzeichnet: 653 Schlüssel
Zuordnungszeilen mit Konfidenzfeld: 4254

=== Unbestätigte Schlüssel mit den meisten Adjektivbelegen ===
  Baum                          24 Belege
  Schmetterling                  9 Belege
  Jungfer                        6 Belege
  Prinzen Samer                  6 Belege
  Nordlichter                  

## Figurenprofile aggregieren => pro Sammlung und korpusweit

In [ ]:
from collections import Counter, defaultdict

def neues_profil():
    return {
        "adjektive": Counter(),          #Lemma -> Häufigkeit
        "vorkommen": [],                 #jede Nennung mit Herkunft
        "maerchen": set(),               #Quellen, in denen die Figur vorkommt
        "gesamt_nennungen": 0,
        "numerus": Counter(),            #"Sing"/"Plur" -> Häufigkeit
        "nennungen_kollektiv": 0,        #Nennungen an einer Gruppe
        "bezug_klassen": Counter(),      #Belegklasse -> Häufigkeit
        "nennungen_bezug_unsicher": 0,   #Nennungen aus nicht belegten Klassen
    }

def fuege_nennung_hinzu(profil, eintrag, quelle, sammlung):
    lemma = (eintrag.get("lemma") or eintrag["adjektiv"]).lower()
    numerus = eintrag.get("numerus")
    kollektiv = bool(eintrag.get("kollektiv"))
    bezug_klasse = eintrag.get("bezug_klasse")
    bezug_sicher = eintrag.get("bezug_sicher")

    profil["adjektive"][lemma] += 1
    profil["vorkommen"].append({
        "adjektiv": eintrag["adjektiv"],
        "lemma": lemma,
        "maerchen": quelle,
        "sammlung": sammlung,
        "satz_id": eintrag["satz_id"],
        "methode": eintrag["methode"],
        "numerus": numerus,
        "kollektiv": kollektiv,
        "coref_sicher": eintrag.get("coref_sicher"),
        "coref_belegklasse": eintrag.get("coref_belegklasse"),
        "bezug_klasse": bezug_klasse,
        "bezug_sicher": bezug_sicher,
    })
    profil["maerchen"].add(quelle)
    profil["gesamt_nennungen"] += 1
    if numerus:
        profil["numerus"][numerus] += 1
    if kollektiv:
        profil["nennungen_kollektiv"] += 1
    if bezug_klasse:
        profil["bezug_klassen"][bezug_klasse] += 1
    if bezug_sicher is False:
        profil["nennungen_bezug_unsicher"] += 1

figuren_profil_aggregiert      = {}                  #figur -> Profil (korpusweit)
figuren_profil_pro_sammlung    = defaultdict(dict)   #sammlung -> figur -> Profil
adjektiv_zuordnungen_alle      = []                  #Einzelzuordnungen (+Herkunft)
figuren_ner_gesamt             = Counter()
figuren_kanonisch_gesamt       = Counter()
figuren_kanonisch_pro_sammlung = defaultdict(Counter)
figuren_bereinigt_gesamt       = []                  #kontextfrei bereinigte Rohformen

for res in ergebnisse_pro_maerchen:
    quelle   = res["quelle"]
    sammlung = res.get("sammlung", "(Wurzel)")

    for name, anzahl in res["figuren_ner"].items():
        figuren_ner_gesamt[name] += anzahl
    for name, anzahl in res["figuren_kanonisch"].items():
        figuren_kanonisch_gesamt[name] += anzahl
        figuren_kanonisch_pro_sammlung[sammlung][name] += anzahl

    for eintrag in res.get("figuren_bereinigt", []):
        figuren_bereinigt_gesamt.append(dict(eintrag, maerchen=quelle, sammlung=sammlung))

    for z in res["zuordnungen"]:
        adjektiv_zuordnungen_alle.append(dict(z, maerchen=quelle, sammlung=sammlung))

    for figur, eintraege in res["figuren_profil"].items():
        agg_gesamt   = figuren_profil_aggregiert.setdefault(figur, neues_profil())
        agg_sammlung = figuren_profil_pro_sammlung[sammlung].setdefault(
            figur, neues_profil())
        for a in eintraege:
            fuege_nennung_hinzu(agg_gesamt,   a, quelle, sammlung)
            fuege_nennung_hinzu(agg_sammlung, a, quelle, sammlung)

figuren_gesamt = sorted(figuren_profil_aggregiert)
figuren_pro_sammlung = {
    sammlung: sorted(profile)
    for sammlung, profile in sorted(figuren_profil_pro_sammlung.items())
}

#Kompakte, JSON-fähige Ansichten (Counter -> sortierte Liste, set -> Liste)
def profil_export_ansicht(agg):
    return {
        "adjektive_haeufigkeit": [
            {"lemma": lemma, "anzahl": n}
            for lemma, n in agg["adjektive"].most_common()
        ],
        "adjektive": sorted(agg["adjektive"]),
        "gesamt_nennungen": agg["gesamt_nennungen"],
        "anzahl_maerchen": len(agg["maerchen"]),
        "maerchen": sorted(agg["maerchen"]),
        "vorkommen": agg["vorkommen"],
        "numerus": dict(agg["numerus"]),
        "nennungen_kollektiv": agg["nennungen_kollektiv"],
        "bezug_klassen": dict(agg["bezug_klassen"]),
        "nennungen_bezug_unsicher": agg["nennungen_bezug_unsicher"],
    }

figuren_profil_export = {
    figur: profil_export_ansicht(agg)
    for figur, agg in figuren_profil_aggregiert.items()
}
figuren_profil_pro_sammlung_export = {
    sammlung: {figur: profil_export_ansicht(agg) for figur, agg in profile.items()}
    for sammlung, profile in sorted(figuren_profil_pro_sammlung.items())
}

#--- Bericht: erst pro Sammlung, dann korpusweit ----------------------------
def _numerus_kurz(agg):
    """Kompakte Numerus-Angabe fuer die Konsolenausgabe."""
    if not agg["numerus"]:
        return ""
    plural = agg["numerus"].get("Plur", 0)
    if not plural:
        return ""
    return f"  [Plur {plural}/{agg['gesamt_nennungen']}]"

for sammlung, profile in sorted(figuren_profil_pro_sammlung.items()):
    n_maerchen = sum(1 for r in ergebnisse_pro_maerchen
                     if r.get("sammlung", "(Wurzel)") == sammlung)
    print(f"=== Sammlung [{sammlung}] — {len(profile)} Figuren "
          f"in {n_maerchen} Märchen ===")
    for figur in sorted(profile,
                        key=lambda f: profile[f]["gesamt_nennungen"], reverse=True):
        agg = profile[figur]
        top = ", ".join(f"{lemma}×{n}" if n > 1 else lemma
                        for lemma, n in agg["adjektive"].most_common(10))
        print(f"  {figur}  ({agg['gesamt_nennungen']} Nennungen, "
              f"{len(agg['maerchen'])} Märchen): {top}{_numerus_kurz(agg)}")
    print()

print(f"=== Korpusweit — {len(figuren_gesamt)} Figuren "
      f"über {len(ergebnisse_pro_maerchen)} Märchen ===")
for figur in sorted(figuren_profil_aggregiert,
                    key=lambda f: figuren_profil_aggregiert[f]["gesamt_nennungen"],
                    reverse=True):
    agg = figuren_profil_aggregiert[figur]
    top = ", ".join(f"{lemma}×{n}" if n > 1 else lemma
                    for lemma, n in agg["adjektive"].most_common(12))
    print(f"  {figur}  ({agg['gesamt_nennungen']} Nennungen, "
          f"{len(agg['maerchen'])} Märchen): {top}{_numerus_kurz(agg)}")

#Übersichtstabelle: eine Zeile pro Sammlung und Figur
df_figuren_sammlung = pd.DataFrame([
    {
        "sammlung": sammlung,
        "figur": figur,
        "nennungen": agg["gesamt_nennungen"],
        "eindeutige_adjektive": len(agg["adjektive"]),
        "maerchen": len(agg["maerchen"]),
        "plural_nennungen": agg["numerus"].get("Plur", 0),
        "kollektiv": agg["nennungen_kollektiv"],
        "top_adjektive": ", ".join(l for l, _ in agg["adjektive"].most_common(5)),
    }
    for sammlung, profile in sorted(figuren_profil_pro_sammlung.items())
    for figur, agg in profile.items()
]).sort_values(["sammlung", "nennungen", "figur"],
               ascending=[True, False, True]).reset_index(drop=True)

#Korpusweite Übersicht plus Spalten 'sammlungen' und 'plural_nennungen'
df_figuren = pd.DataFrame([
    {
        "figur": figur,
        "nennungen": agg["gesamt_nennungen"],
        "eindeutige_adjektive": len(agg["adjektive"]),
        "maerchen": len(agg["maerchen"]),
        "sammlungen": len({v["sammlung"] for v in agg["vorkommen"]}),
        "plural_nennungen": agg["numerus"].get("Plur", 0),
        "kollektiv": agg["nennungen_kollektiv"],
        "top_adjektive": ", ".join(l for l, _ in agg["adjektive"].most_common(5)),
    }
    for figur, agg in figuren_profil_aggregiert.items()
]).sort_values(["nennungen", "figur"], ascending=[False, True]).reset_index(drop=True)

df_figuren_sammlung

=== Sammlung [Arabische Märchen] — 168 Figuren in 8 Märchen ===
  Prinz  (84 Nennungen, 2 Märchen): lang×4, alt×3, glücklich×3, anderer×3, voll×2, leicht×2, bereit×2, augenblicklich×2, bekannt×2, zufrieden×2  [Plur 2/84]
  Mann  (48 Nennungen, 7 Märchen): tot×3, jung×3, bereit×2, sanft×2, weiser×2, alt×2, reich×2, richtig×2, willfährig, groß  [Plur 6/48]
  Ali Baba  (44 Nennungen, 1 Märchen): genau×2, sorgfältig×2, ungeduldig×2, reich×2, gut×2, vorsichtig×2, leise×2, anderer, dürftig, aufmerksam
  Frau  (43 Nennungen, 5 Märchen): schön×7, gut×5, gnädig×2, anderer×2, lang×2, klein×2, vollkommenste, wahrhaftig, treu, mächtig  [Plur 8/43]
  Geist  (40 Nennungen, 3 Märchen): bös×4, treu×4, lang×3, schlimm×2, böse×2, liebevoll, schrecklich, häßlich, abscheulich, abtrünnig  [Plur 3/40]
  Mädchen  (39 Nennungen, 6 Märchen): schön×7, fremd×2, zart×2, reizend×2, lieblich×2, wunderbar, ungefähr, langsam, achtundneunzig, tugendhaft  [Plur 14/39]
  König  (38 Nennungen, 5 Märchen): groß×3, reich×2

,sammlung,figur,nennungen,eindeutige_adjektive,maerchen,plural_nennungen,kollektiv,top_adjektive
0,Arabische Märchen,Prinz,84,63,2,2,2,"lang, alt, glücklich, anderer, voll"
1,Arabische Märchen,Mann,48,38,7,6,6,"tot, jung, bereit, sanft, weiser"
2,Arabische Märchen,Ali Baba,44,37,1,0,0,"genau, sorgfältig, ungeduldig, reich, gut"
3,Arabische Märchen,Frau,43,29,5,8,8,"schön, gut, gnädig, anderer, lang"
4,Arabische Märchen,Geist,40,30,3,3,3,"bös, treu, lang, schlimm, böse"
...,...,...,...,...,...,...,...,...
695,Japanische Märchen,Verbeugungen,1,1,1,0,1,tief
696,Japanische Märchen,Verwandten,1,1,1,1,1,wohnend
697,Japanische Märchen,Volk,1,1,1,0,0,ganz
698,Japanische Märchen,Wohlleben,1,1,1,0,0,behaglich


## Schritt 11 — Normalisierung des Adjektivvokabulars


In [ ]:
#===========================================================================
#Schritt 11 — Normalisierung des Adjektivvokabulars
#===========================================================================

try:
    from pipeline.adjektiv_normalisierung import normalisiere_adjektive
    NORMALISIERUNG_AKTIV = True
except ImportError as fehler:
    NORMALISIERUNG_AKTIV = False
    print("Modul pipeline.adjektiv_normalisierung nicht gefunden — die "
          "Normalisierung bleibt ohne Wirkung.")
    print(f"  ({fehler})")
    print("Das Feld 'lemma_norm' entsteht dennoch und traegt dann das Lemma "
          "selbst; 'normalisierung.aktiv' steht im Export auf false.")

if NORMALISIERUNG_AKTIV:
    normalisierung = normalisiere_adjektive(adjektiv_zuordnungen_alle)
    norm_karte = normalisierung["karte"]
    print(f"Regelwerk {normalisierung['meta']['regelwerk']} "
          f"(Cistem: {normalisierung['meta']['cistem_quelle']})")
    print(f"  Vokabular          : {normalisierung['meta']['vokabular']} Lemmata")
    print(f"  Klassen            : {normalisierung['meta']['klassen']}")
    print(f"  Zusammenfuehrungen : {normalisierung['meta']['regeln']}")
    print(f"  gesperrte Paare    : {normalisierung['meta']['paare_gesperrt']}")
    print(f"  Endungsinventar    : {normalisierung['meta']['endungsinventar']}")
    if normalisierung["paare_gesperrt"]:
        print("  Sperren im Einzelnen:")
        for p in normalisierung["paare_gesperrt"]:
            print(f"    {p['a']} / {p['b']}  [{p['sperre']}]")
else:
    normalisierung = {"karte": {}, "klassen": [], "paare_vereinigt": [],
                      "paare_gesperrt": [],
                      "meta": {"verfahren": "regelbasiert", "regelwerk": None,
                               "cistem_quelle": None, "parameter": {},
                               "endungsinventar": {}, "vokabular": 0,
                               "klassen": 0, "regeln": 0,
                               "paare_vereinigt": 0, "paare_gesperrt": 0}}
    norm_karte = {}

#Signale und Normformregel je Lemma für die Exportkarte bereitstellen
signale_je_lemma = {}
normform_regel_je_klasse = {}
for klasse in normalisierung["klassen"]:
    normform_regel_je_klasse[klasse["normform"]] = klasse["normform_regel"]
    for mitglied in klasse["mitglieder"]:
        signale_je_lemma[mitglied["lemma"]] = ", ".join(mitglied["signale"])


def normiere_lemma(lemma):
    """Normform eines Adjektivlemmas. Ohne Regel bleibt das Lemma unveraendert."""
    kleingeschrieben = (lemma or "").lower()
    return norm_karte.get(kleingeschrieben, kleingeschrieben)


#--- 1. Zuordnungszeilen ---------------------------------------------------
zeilen_veraendert = 0
for z in adjektiv_zuordnungen_alle:
    z["lemma_norm"] = normiere_lemma(z.get("lemma"))
    if z["lemma_norm"] != (z.get("lemma") or "").lower():
        zeilen_veraendert += 1

#--- 2. Profil-Vorkommen auf beiden Ebenen --------------------------------
_alle_profile = list(figuren_profil_aggregiert.values()) + [
    agg for profile in figuren_profil_pro_sammlung.values()
    for agg in profile.values()
]
for _agg in _alle_profile:
    _agg["adjektive_norm"] = Counter()
    for _vorkommen in _agg["vorkommen"]:
        _vorkommen["lemma_norm"] = normiere_lemma(_vorkommen.get("lemma"))
        _agg["adjektive_norm"][_vorkommen["lemma_norm"]] += 1


def _ergaenze_profil_ansicht(ansicht, agg):
    """Ergaenzt eine bestehende Export-Ansicht additiv um die Normform-Zaehler."""
    ansicht["adjektive_norm_haeufigkeit"] = [
        {"normform": normform, "anzahl": anzahl}
        for normform, anzahl in agg["adjektive_norm"].most_common()
    ]
    ansicht["adjektive_norm"] = sorted(agg["adjektive_norm"])


for _figur, _agg in figuren_profil_aggregiert.items():
    _ergaenze_profil_ansicht(figuren_profil_export[_figur], _agg)
for _sammlung, _profile in figuren_profil_pro_sammlung.items():
    for _figur, _agg in _profile.items():
        _ergaenze_profil_ansicht(
            figuren_profil_pro_sammlung_export[_sammlung][_figur], _agg)

#--- 3. Ergebnis pro Märchen ----------------------------------------------
for _res in ergebnisse_pro_maerchen:
    for _eintraege in _res.get("figuren_profil", {}).values():
        for _eintrag in _eintraege:
            _eintrag["lemma_norm"] = normiere_lemma(_eintrag.get("lemma"))
    for _z in _res.get("zuordnungen", []):
        _z["lemma_norm"] = normiere_lemma(_z.get("lemma"))

#--- 4. Karte Normform -> Lemmata mit Häufigkeiten ------------------------
_zugeordnete = [z for z in adjektiv_zuordnungen_alle
                if z["status"] == "zugeordnet" and z.get("lemma")]
adjektiv_lemma_haeufigkeit = Counter(z["lemma"].lower() for z in _zugeordnete)
adjektiv_norm_haeufigkeit = Counter(z["lemma_norm"] for z in _zugeordnete)

_norm_lemmata = defaultdict(Counter)
_norm_maerchen = defaultdict(set)
_norm_figuren = defaultdict(set)
for z in _zugeordnete:
    _norm_lemmata[z["lemma_norm"]][z["lemma"].lower()] += 1
    _norm_maerchen[z["lemma_norm"]].add(z["maerchen"])
    if z.get("figur"):
        _norm_figuren[z["lemma_norm"]].add(z["figur"])

adjektiv_norm_karte = {
    normform: {
        "belege": adjektiv_norm_haeufigkeit[normform],
        "anzahl_lemmata": len(lemmata),
        "normform_regel": normform_regel_je_klasse.get(normform),
        "lemmata": [
            {
                "lemma": lemma,
                "anzahl": anzahl,
                "signale": signale_je_lemma.get(lemma, ""),
            }
            for lemma, anzahl in lemmata.most_common()
        ],
        "anzahl_maerchen": len(_norm_maerchen[normform]),
        "anzahl_figuren": len(_norm_figuren[normform]),
        "zusammengefuehrt": len(lemmata) > 1 or normform not in lemmata,
    }
    for normform, lemmata in sorted(_norm_lemmata.items())
}

#--- 5. Kennzahlen und Abgleiche -------------------------------------------
_vokabular = set(adjektiv_lemma_haeufigkeit)
_normformen_neu = sorted({n for n in norm_karte.values() if n not in _vokabular})
_zusammengefuehrt = [n for n, e in adjektiv_norm_karte.items()
                     if e["anzahl_lemmata"] > 1]

normalisierung_bilanz = {
    "aktiv": NORMALISIERUNG_AKTIV,
    **normalisierung["meta"],
    "normformen_ohne_eigenes_lemma": _normformen_neu,
    "zeilen_veraendert": zeilen_veraendert,
    "vokabular_lemma": len(adjektiv_lemma_haeufigkeit),
    "vokabular_normform": len(adjektiv_norm_haeufigkeit),
    "normformen_mit_zusammenfuehrung": len(_zusammengefuehrt),
}

print()
print("Wirkung der Normalisierung (Grundgesamtheit: Status 'zugeordnet')")
print(f"  Vokabular vorher   : {len(adjektiv_lemma_haeufigkeit)} Lemmata")
print(f"  Vokabular nachher  : {len(adjektiv_norm_haeufigkeit)} Normformen "
      f"(-{len(adjektiv_lemma_haeufigkeit) - len(adjektiv_norm_haeufigkeit)})")
print(f"  Normformen mit Zusammenführung: {len(_zusammengefuehrt)}")
print(f"  Zeilen mit abweichender Normform: {zeilen_veraendert} von "
      f"{len(adjektiv_zuordnungen_alle)}")
_einmal_vorher = sum(1 for n in adjektiv_lemma_haeufigkeit.values() if n == 1)
_einmal_nachher = sum(1 for n in adjektiv_norm_haeufigkeit.values() if n == 1)
print(f"  Einmalbelege       : {_einmal_vorher} -> {_einmal_nachher}")

if _normformen_neu:
    print(f"\n  Hinweis: {len(_normformen_neu)} Normformen treten neu auf, ohne selbst "
          f"als Lemma belegt zu sein: {', '.join(_normformen_neu)}")
    print("    Wer von 'lemma_norm' auf 'lemma' zurückschließen will, muss dafür "
          "'adjektiv_norm_karte' heranziehen.")

print("\nDie zehn belegstärksten Zusammenführungen:")
for normform in sorted(_zusammengefuehrt,
                       key=lambda n: -adjektiv_norm_karte[n]["belege"])[:10]:
    eintrag = adjektiv_norm_karte[normform]
    teile = ", ".join(f"{t['lemma']} ({t['anzahl']})" for t in eintrag["lemmata"])
    print(f"  {normform:<16} {eintrag['belege']:>4} Belege  <- {teile}")


Regelwerk AP6-R2 (Cistem: nltk)
  Vokabular          : 958 Lemmata
  Klassen            : 45
  Zusammenfuehrungen : 57
  gesperrte Paare    : 1
  Endungsinventar    : {'e': 850, 'en': 604, 'er': 338, 'es': 139, 'sten': 30, 'ste': 17, 'em': 16, 'n': 9, 'ere': 5}
  Sperren im Einzelnen:
    geschickt / schicken  [ge_praefix]

Wirkung der Normalisierung (Grundgesamtheit: Status 'zugeordnet')
  Vokabular vorher   : 958 Lemmata
  Vokabular nachher  : 901 Normformen (-57)
  Normformen mit Zusammenführung: 45
  Zeilen mit abweichender Normform: 597 von 11022
  Einmalbelege       : 551 -> 508

Die zehn belegstärksten Zusammenführungen:
  klein             259 Belege  <- klein (255), kleine (2), kleiner (1), kleines (1)
  alte              229 Belege  <- alt (226), alte (3)
  lieb              123 Belege  <- lieb (116), liebes (3), liebe (2), liebste (1), liebst (1)
  arm               108 Belege  <- arm (105), armer (1), armes (1), ärm (1)
  jung               96 Belege  <- jung (94), jüng (2)

## Ergebnis exportieren

In [ ]:
#===========================================================================
# Schritt 12 — Instanzprofile normalisieren und Abdeckung
#===========================================================================

_lemmafeld = ("lemma_norm"
              if any("lemma_norm" in z for z in adjektiv_zuordnungen_alle)
              else "lemma")

_angeglichen = sum(1 for z in adjektiv_zuordnungen_alle
                   if z.get("figur_rolle") != z.get("figur"))
for z in adjektiv_zuordnungen_alle:
    z["figur_rolle"] = z.get("figur")
for _res in ergebnisse_pro_maerchen:
    for z in _res.get("zuordnungen", []):
        z["figur_rolle"] = z.get("figur")
print(f"figur_rolle an den Nachlauf angeglichen: {_angeglichen} Zeilen")

for _res in ergebnisse_pro_maerchen:
    _zeilen = [z for z in adjektiv_zuordnungen_alle
               if z.get("maerchen") == _res["quelle"]]
    _res["instanzprofile"] = inst.baue_instanzprofile(_zeilen, _lemmafeld)
    _res["instanz_abdeckung"] = inst.abdeckungsbericht(_zeilen)

#--- korpusweite Abdeckung ------------------------------------------------
instanz_abdeckung_gesamt = inst.abdeckungsbericht(adjektiv_zuordnungen_alle)

_zug = [z for z in adjektiv_zuordnungen_alle if z.get("status") == "zugeordnet"]
_zugewiesen = [z for z in _zug if z.get("figur_instanz")]
_verdacht = [z for z in _zug if z.get("instanz_verdacht")]
_verdacht_zerfall = [z for z in _zug if z.get("instanz_verdacht_zerfall")]

print(f"Instanzprofile neu gebaut ueber '{_lemmafeld}'")
print(f"  Zuordnungen mit Figur   : {len(_zug)}")
print(f"  Instanz zugewiesen      : {len(_zugewiesen)} "
      f"({100 * len(_zugewiesen) / max(len(_zug), 1):.2f} %)")
print(f"  verdacht (Oberflaeche)  : {len(_verdacht)} "
      f"({100 * len(_verdacht) / max(len(_zug), 1):.1f} %)")
print(f"  verdacht (Coref-Zerfall): {len(_verdacht_zerfall)} "
      f"({100 * len(_verdacht_zerfall) / max(len(_zug), 1):.1f} %, getrennt "
      f"gefuehrt, zaehlt seit Modulfassung V2 nicht fuer den Verdacht)")
print(f"  Instanzen gesamt        : "
      f"{len({(z['maerchen'], z['figur_instanz']) for z in _zugewiesen})}")


figur_rolle an den Nachlauf angeglichen: 4254 Zeilen
Instanzprofile neu gebaut ueber 'lemma_norm'
  Zuordnungen mit Figur   : 4254
  Instanz zugewiesen      : 49 (1.15 %)
  verdacht (Oberflaeche)  : 659 (15.5 %)
  verdacht (Coref-Zerfall): 2229 (52.4 %, getrennt gefuehrt, zaehlt seit Modulfassung V2 nicht fuer den Verdacht)
  Instanzen gesamt        : 40


In [ ]:
korpus_uebersicht = [
    {
        "quelle": res["quelle"],
        "sammlung": res.get("sammlung", "(Wurzel)"),
        "titel": res.get("titel"),
        **res["statistik"],
        "coref_ok": res["coref_ok"],
        "coref_protokoll_datei": res.get("coref_protokoll_datei"),
    }
    for res in ergebnisse_pro_maerchen
]

#Übersicht der Sammlungen: welche Märchen gehören zu welcher Sammlung?
sammlungen_uebersicht = {}
for res in ergebnisse_pro_maerchen:
    s = res.get("sammlung", "(Wurzel)")
    eintrag = sammlungen_uebersicht.setdefault(
        s, {"anzahl_maerchen": 0, "maerchen": []})
    eintrag["anzahl_maerchen"] += 1
    eintrag["maerchen"].append(res.get("titel") or res["quelle"])
sammlungen_uebersicht = dict(sorted(sammlungen_uebersicht.items()))

#Schlüsselkarte => macht die Kanonisierung von aussen nachvollziehbarer
figuren_schluessel_karte = {
    schluessel: {
        "anzeigename": anzeigename_von[schluessel],
        "formen": dict(formen.most_common()),
        "gesamt": sum(formen.values()),
        "anzahl_formen": len(formen),
    }
    for schluessel, formen in sorted(formen_je_schluessel.items())
}

bezug_klassen_gesamt = Counter(
    z.get("bezug_klasse") or "(ohne)"
    for z in adjektiv_zuordnungen_alle if z["status"] == "zugeordnet"
)
bezug_regeln_gesamt = Counter(
    z.get("bezug_regel") or "(ohne)"
    for z in adjektiv_zuordnungen_alle if z["status"] == "zugeordnet"
)

gesperrte_belege = [z for z in adjektiv_zuordnungen_alle
                    if z.get("bezug_gesperrt")]
gesperrt_je_sammlung = Counter(z.get("sammlung", "(Wurzel)")
                               for z in gesperrte_belege)

ergebnis = {
    "meta": { #Metadaten etablieren
        "notebook_version": "V1.0",
        "normalisierung": normalisierung_bilanz,
        "anzahl_schluessel": len(figuren_schluessel_karte),
        "anzahl_verworfen": len(nachlauf_protokoll["verworfene_schluessel"]),
        "anzahl_kollisionen": len(nachlauf_protokoll["kollisionen"]),
    },
    "korpus": {
        "anzahl_maerchen": len(ergebnisse_pro_maerchen),
        "anzahl_sammlungen": len(sammlungen_uebersicht),
        "anzahl_fehlgeschlagen": len(fehler_maerchen),
        "sammlungen": sammlungen_uebersicht,
        "maerchen": korpus_uebersicht,
        "fehlgeschlagen": fehler_maerchen,
    },
    #korpusweit
    "figuren_gesamt": figuren_gesamt,
    "figuren_ner_haeufigkeit_gesamt": dict(figuren_ner_gesamt),
    "figuren_kanonisch_gesamt": dict(figuren_kanonisch_gesamt),
    "figuren_profil_aggregiert": figuren_profil_export,
    #aufgeteilt und aggregiert pro Sammlung
    "figuren_pro_sammlung": figuren_pro_sammlung,
    "figuren_kanonisch_pro_sammlung": {
        s: dict(c) for s, c in sorted(figuren_kanonisch_pro_sammlung.items())
    },
    "figuren_profil_pro_sammlung": figuren_profil_pro_sammlung_export,
    #Einzelzuordnungen + unaggregierte Ergebnisse pro Märchen
    "adjektiv_zuordnungen_alle": adjektiv_zuordnungen_alle,
    "pro_maerchen": [
        {
            "quelle": res["quelle"],
            "sammlung": res.get("sammlung", "(Wurzel)"),
            "titel": res.get("titel"),
            "figuren": res["figuren"],
            "figuren_kanonisch": res["figuren_kanonisch"],
            "satz_mapping": res["satz_mapping"],
            "figuren_profil": res["figuren_profil"],
            "statistik": res["statistik"],
            "coref_protokoll_datei": res.get("coref_protokoll_datei"), 
            "figuren_bereinigt": res.get("figuren_bereinigt", []),
            "schluessel_formen": res.get("schluessel_formen", {}),
            "figuren_profil_instanzen": res.get("instanzprofile", {}),
            "instanz_abdeckung": res.get("instanz_abdeckung", {}),
            "instanz_plausibilitaet": res.get("instanz_plausibilitaet", []),
            "instanz_verdacht_befunde": res.get("instanz_verdacht_befunde", {}),
        }
        for res in ergebnisse_pro_maerchen
    ],
    "figuren_schluessel_karte": figuren_schluessel_karte,
    "figuren_belebtheit": {
        schluessel: {
            "anzeigename": anzeigename_von.get(schluessel),
            "sicher": eintrag["sicher"],
            "anzahl_signale": eintrag["anzahl_signale"],
            "grund": eintrag["grund"],
            "signale": eintrag["signale"],
            "treffer": eintrag["treffer"],
            "einteilig": eintrag["einteilig"],
        }
        for schluessel, eintrag in sorted(belebtheit.items())
    },
    "bezug_klassen_gesamt": dict(bezug_klassen_gesamt),
    "bezug_regeln_gesamt": dict(bezug_regeln_gesamt),
    "adjektiv_norm_karte": adjektiv_norm_karte,
    "instanz_abdeckung_gesamt": instanz_abdeckung_gesamt,
    "gesperrte_belege_gesamt": len(gesperrte_belege),
    "gesperrte_belege_je_sammlung": dict(sorted(gesperrt_je_sammlung.items())),
    "figuren_bereinigt_gesamt": figuren_bereinigt_gesamt,
    "nachlauf_protokoll": nachlauf_protokoll,
}

with open("prototyp_ergebnisse_alle.json", "w", encoding="utf-8") as f:
    json.dump(ergebnis, f, ensure_ascii=False, indent=2)
print(f"Gespeichert: prototyp_ergebnisse_alle.json "
      f"({len(adjektiv_zuordnungen_alle)} Zuordnungen, {len(figuren_gesamt)} Figuren, "
      f"{len(ergebnisse_pro_maerchen)} Märchen in "
      f"{len(sammlungen_uebersicht)} Sammlung(en))")
_zugeordnet = sum(bezug_klassen_gesamt.values())
_unsicher = sum(
    1 for z in adjektiv_zuordnungen_alle
    if z["status"] == "zugeordnet" and z.get("bezug_sicher") is False
)
print(f"  Belegklassen: " + ", ".join(
    f"{k} {n}" for k, n in bezug_klassen_gesamt.most_common()))
print(f"  davon aus nicht belegten Klassen: {_unsicher} von {_zugeordnet} "
      f"({_unsicher / max(1, _zugeordnet):.1%})")
#Bilanz der Sperre
if SPERRE_ADVERBIAL:
    _gesamt_belege = len(adjektiv_zuordnungen_alle)
    print(f"  gesperrt (adverbial ohne Figurenbezug): {len(gesperrte_belege)} "
          f"Belege, unter den offenen auffindbar")
    print(f"  Zuordnungsquote: {_zugeordnet}/{_gesamt_belege} = "
          f"{_zugeordnet / max(1, _gesamt_belege):.1%}")
print(f"  zusätzlich: {len(figuren_schluessel_karte)} Schlüssel in der Schlüsselkarte, "
      f"{len(nachlauf_protokoll['verworfene_schluessel'])} verworfene Schlüssel und "
      f"{len(konsolidierungsprotokoll)} Dativ-Zusammenführungen protokolliert")

#--- Adjektiv-Export ---
adjektive_lemmata = sorted({
    z["lemma"].lower()
    for z in adjektiv_zuordnungen_alle
    if z["status"] == "zugeordnet" and z["lemma"]
})
with open("adjektive_fuer_clustering.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(adjektive_lemmata))

adjektive_mit_figur = sorted({
    (z["lemma"].lower(), z["figur"])
    for z in adjektiv_zuordnungen_alle
    if z["status"] == "zugeordnet" and z["lemma"] and z["figur"]
})
with open("adjektive_fuer_clustering.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "adjektive": adjektive_lemmata,
            "adjektive_mit_figur": [
                {"adjektiv": a, "figur": fig} for a, fig in adjektive_mit_figur
            ],
        },
        f, ensure_ascii=False, indent=2,
    )
print(f"Gespeichert: adjektive_fuer_clustering.txt / .json "
      f"({len(adjektive_lemmata)} eindeutige Lemmata korpusweit)")

adjektive_norm = sorted(adjektiv_norm_karte)
with open("adjektive_norm_fuer_clustering.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(adjektive_norm))

norm_figur_haeufigkeit = Counter(
    (z["lemma_norm"], z["figur"])
    for z in adjektiv_zuordnungen_alle
    if z["status"] == "zugeordnet" and z.get("lemma") and z.get("figur")
)
with open("adjektive_norm_fuer_clustering.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "adjektive": adjektive_norm,
            "adjektive_mit_figur": [
                {"adjektiv": normform, "figur": figur, "anzahl": anzahl}
                for (normform, figur), anzahl in sorted(norm_figur_haeufigkeit.items())
            ],
            "haeufigkeit_normform": [
                {
                    "normform": normform,
                    "anzahl": eintrag["belege"],
                    "anzahl_lemmata": eintrag["anzahl_lemmata"],
                    "anzahl_maerchen": eintrag["anzahl_maerchen"],
                    "anzahl_figuren": eintrag["anzahl_figuren"],
                    "lemmata": eintrag["lemmata"],
                }
                for normform, eintrag in sorted(
                    adjektiv_norm_karte.items(), key=lambda p: (-p[1]["belege"], p[0]))
            ],
            "haeufigkeit_lemma": [
                {"lemma": lemma, "normform": normiere_lemma(lemma), "anzahl": anzahl}
                for lemma, anzahl in sorted(
                    adjektiv_lemma_haeufigkeit.items(), key=lambda p: (-p[1], p[0]))
            ],
            "meta": normalisierung_bilanz,
        },
        f, ensure_ascii=False, indent=2,
    )
print(f"Gespeichert: adjektive_norm_fuer_clustering.txt / .json "
      f"({len(adjektive_norm)} Normformen korpusweit, davon "
      f"{normalisierung_bilanz['normformen_mit_zusammenfuehrung']} zusammengeführt)")

Gespeichert: prototyp_ergebnisse_alle.json (11022 Zuordnungen, 550 Figuren, 80 Märchen in 4 Sammlung(en))
  Belegklassen: attributiv 2093, artangabe 1197, praedikativ 779, relativsatz 99, objektspraedikativ 86
  davon aus nicht belegten Klassen: 1296 von 4254 (30.5%)
  gesperrt (adverbial ohne Figurenbezug): 307 Belege, unter den offenen auffindbar
  Zuordnungsquote: 4254/11022 = 38.6%
  zusätzlich: 1090 Schlüssel in der Schlüsselkarte, 1 verworfene Schlüssel und 71 Dativ-Zusammenführungen protokolliert
Gespeichert: adjektive_fuer_clustering.txt / .json (958 eindeutige Lemmata korpusweit)
Gespeichert: adjektive_norm_fuer_clustering.txt / .json (901 Normformen korpusweit, davon 45 zusammengeführt)


## Nachnutzung / Abfragen

In [33]:
import json

with open("prototyp_ergebnisse_alle.json", "r", encoding="utf-8") as f:
    daten = json.load(f)

figur_gesucht = "Stiefmutter"   # anpassen (siehe daten["figuren_pro_sammlung"])

treffer = {
    sammlung: profile[figur_gesucht]
    for sammlung, profile in daten["figuren_profil_pro_sammlung"].items()
    if figur_gesucht in profile
}

if not treffer:
    print(f"Figur '{figur_gesucht}' nicht im Korpus. Verfügbare Figuren je Sammlung:")
    for sammlung, figuren in daten["figuren_pro_sammlung"].items():
        print(f"  [{sammlung}] {', '.join(figuren)}")
else:
    for sammlung, profil in sorted(treffer.items()):
        print(f"[{sammlung}] {figur_gesucht}: {profil['gesamt_nennungen']} Nennungen "
              f"in {profil['anzahl_maerchen']} Märchen")
        for eintrag in profil["adjektive_haeufigkeit"]:
            print(f"    {eintrag['lemma']}: {eintrag['anzahl']}×")
        print()
    gesamt = daten["figuren_profil_aggregiert"][figur_gesucht]
    print(f"Korpusweit: {gesamt['gesamt_nennungen']} Nennungen "
          f"in {gesamt['anzahl_maerchen']} Märchen "
          f"über {len(treffer)} Sammlung(en)")

[Brüder Grimm] Stiefmutter: 6 Nennungen in 4 Märchen
    bös: 2×
    rein: 1×
    bleich: 1×
    heftig: 1×
    gottlos: 1×

[H. C. Andersen] Stiefmutter: 1 Nennungen in 1 Märchen
    bös: 1×

Korpusweit: 7 Nennungen in 5 Märchen über 2 Sammlung(en)


In [ ]:
from collections import Counter

#Wie oft wurde welcher Figur ein Adjektiv zugeordnet => je Sammlung & gesamt?
zaehler_pro_sammlung = {}
zaehler_gesamt = Counter()

for eintrag in daten["adjektiv_zuordnungen_alle"]:
    if not eintrag.get("figur"):
        continue
    sammlung = eintrag.get("sammlung", "(Wurzel)")
    zaehler_pro_sammlung.setdefault(sammlung, Counter())[eintrag["figur"]] += 1
    zaehler_gesamt[eintrag["figur"]] += 1

for sammlung, counter in sorted(zaehler_pro_sammlung.items()):
    print(f"[{sammlung}]")
    for figur, anzahl in counter.most_common():
        print(f"  {figur}: {anzahl}")
    print()

print("[Korpusweit]")
for figur, anzahl in zaehler_gesamt.most_common():
    print(f"  {figur}: {anzahl}")

[Arabische Märchen]
  Prinz: 84
  Mann: 48
  Ali Baba: 44
  Frau: 43
  Geist: 40
  Mädchen: 39
  König: 38
  Gott: 32
  Fischer: 30
  Perser: 29
  Sklavin Morgiana: 29
  Sultan: 27
  Prinzessin: 27
  Räuber: 27
  Fisch: 26
  Sklave: 25
  Bruder: 22
  Herr: 22
  Casim: 21
  Fee: 15
  Weise: 14
  Zauberin: 13
  Sohn: 12
  Chodadad: 12
  Jüngling: 11
  Mustapha: 11
  Scheherban: 10
  Hauptmann: 10
  Apfel: 9
  Zauberer: 9
  Leute: 9
  Geliebter: 9
  Diener: 8
  Kaufmann: 8
  Kaufleute: 7
  Tod: 7
  Fräulein: 7
  Kunde: 7
  Tier: 7
  Herrin: 7
  Weib: 6
  Wesir: 6
  Reiter: 6
  Prinzen Samer: 6
  Kaiser: 6
  Kind: 5
  Tochter: 5
  Freund: 5
  Pferd: 5
  Mensch: 5
  Baum: 5
  Schwester Dinarsad: 4
  Günstling: 4
  Esel: 4
  Grieche: 4
  Vetter: 4
  Gestalt: 4
  Nichte Nurunnihar: 3
  Volk: 3
  Löwen: 3
  Bruder Schaïbar: 3
  Knebelbart: 3
  Schwester: 3
  Schicksal: 3
  Abessinier: 3
  Gazelle: 3
  Sklavin: 3
  Gärtner: 3
  Dieb: 3
  Dinarsad: 2
  Vater: 2
  Ausrufer: 2
  Karawane: 2
  Köch

In [ ]:
#Nach Verlässlichkeit der Zuordnung filtern

from collections import Counter

klassen, unsicher = Counter(), Counter()
for eintrag in daten["adjektiv_zuordnungen_alle"]:
    if eintrag["status"] != "zugeordnet":
        continue
    klasse = eintrag.get("bezug_klasse") or "(ohne)"
    klassen[klasse] += 1
    if eintrag.get("bezug_sicher") is False:
        unsicher[klasse] += 1

gesamt = sum(klassen.values())
print(f"Zugeordnete Belege: {gesamt}")
for klasse, anzahl in klassen.most_common():
    print(f"  {klasse:<30} {anzahl:>5} ({anzahl / max(1, gesamt):5.1%})"
          f"   davon nicht belegt: {unsicher[klasse]:>5}")

#Dasselbe Figurenprofil einmal vollständig und einmal nur aus belegten Klassen
figur_gesucht = "Stiefmutter"   # anpassen
alle_lemmata, sichere_lemmata = Counter(), Counter()
for eintrag in daten["adjektiv_zuordnungen_alle"]:
    if eintrag["status"] != "zugeordnet" or eintrag.get("figur") != figur_gesucht:
        continue
    lemma = (eintrag.get("lemma") or eintrag["adjektiv"]).lower()
    alle_lemmata[lemma] += 1
    if eintrag.get("bezug_sicher") is True:
        sichere_lemmata[lemma] += 1

print(f"\nProfil '{figur_gesucht}': {sum(alle_lemmata.values())} Nennungen "
      f"gesamt, {sum(sichere_lemmata.values())} aus belegten Klassen")
for lemma, anzahl in alle_lemmata.most_common(15):
    print(f"    {lemma:<20} {anzahl:>3}x   belegt: {sichere_lemmata[lemma]:>3}x")

Zugeordnete Belege: 4254
  attributiv                      2093 (49.2%)   davon nicht belegt:     0
  artangabe                       1197 (28.1%)   davon nicht belegt:  1197
  praedikativ                      779 (18.3%)   davon nicht belegt:     0
  relativsatz                       99 ( 2.3%)   davon nicht belegt:    99
  objektspraedikativ                86 ( 2.0%)   davon nicht belegt:     0

Profil 'Stiefmutter': 7 Nennungen gesamt, 6 aus belegten Klassen
    bös                    3x   belegt:   3x
    rein                   1x   belegt:   1x
    bleich                 1x   belegt:   1x
    heftig                 1x   belegt:   0x
    gottlos                1x   belegt:   1x
